# 第 56 课｜多次采样、Self-consistency 与 Best-of-N

**从“生成一个答案”走向“生成候选、选择答案、核算代价”。**

本课承接第 55 课：冻结同一套 4 道开发题、12 道评测题及评分规则，把每题扩展到多个预先登记的候选槽。

> 默认实验是**手工候选夹具**，不调用模型、不下载权重、不更新参数。它用于检查工程和统计方法，不是模型能力成绩。真实 Qwen 采样入口默认关闭；制作环境没有 Transformers，未端到端实测。

| 完成后应能解释 | 必须完成的实践 |
|---|---|
| 候选覆盖与最终选对的区别 | 先保存选择，再用私有目标评分 |
| 规范化投票与代理评分的差异 | 相同候选池运行三个选择器 |
| `pass@k` 不是部署答案 | 分开记录覆盖率、选择损失与成本 |
| 多生成不保证更准确 | 重现错误多数、公开测试盲点和 Best-of-N 退步 |

建议分两次学习：第一次完成投票与候选选择，第二次完成成本、早停和独立练习。CPU 足够；默认仅 Python 标准库。

## 0. 使用方法与范围

从上到下运行，`Shift + Enter` 执行当前单元。实现代码已经完整嵌入，**不依赖同名 `.py` 或第 55 课文件**；标记为“实现”的较长单元可折叠后阅读。

本课区分三类信息：

- **生成输入**：问题、格式要求、公开代码样例。
- **选择输入**：以上公共信息＋实际候选，不包含参考答案或额外评测测试。
- **事后评分**：在选择锁定后，才使用参考值和留出测试。

这是一项代码契约，不是针对拥有源码读取权限的人的安全隔离。课堂答案和测试公开，不声称是保密基准。

后面的独立练习默认不执行，填好后再打开开关。运行所有单元成功，不等于完成了练习。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第56课：多次采样、答案投票与 Best-of-N。

Python 3.10+；默认仅标准库、离线、不调用模型、不训练、不执行任意 Python。
--sample 可选加载真实 Qwen；需额外 torch/Transformers，制作环境未端到端实测。
有限 AST 解释器用于课堂指定的 solve(numbers) 子集，不是通用安全沙箱。
内置测试和答案公开：留出仅指不进入生成请求，不声称防泄漏或生产认证。
"""
from __future__ import annotations

In [2]:
import argparse
import ast
import copy
import hashlib
import itertools
import json
import math
import platform
import re
import sys
import random
import tempfile
import time
from collections import Counter
from contextlib import contextmanager
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

VERSION = "lesson56-selection-v1"
MAX_RESPONSE_BYTES = 16384
MAX_SOURCE_BYTES = 4096
MAX_AST_NODES = 300
MAX_AST_DEPTH = 24
MAX_STEPS = 2000
MAX_INTEGER = 10**9
MAX_INPUT_LEN = 16
MAX_FILE_BYTES = 16 * 1024 * 1024
MAX_RECORDS = 2048
NUMBER_RE = re.compile(r"[+-]?[0-9]+(?:/[0-9]+|\.[0-9]+)?\Z")
SOURCES = {
    "fractions": "https://docs.python.org/3.13/library/fractions.html",
    "json": "https://docs.python.org/3.13/library/json.html",
    "ast": "https://docs.python.org/3.13/library/ast.html",
    "verifiers": "https://arxiv.org/abs/2110.14168",
    "process_outcome": "https://arxiv.org/abs/2305.20050",
    "code_warning": "https://github.com/openai/human-eval/blob/master/human_eval/execution.py",
}


class LessonError(ValueError):
    """有界输入或协议不满足本课约定。"""


class UnsupportedCode(LessonError):
    """超出课堂语言范围，不等于该代码在完整 Python 中语义错误。"""


class RuntimeLimit(LessonError):
    """有限解释器的运行预算耗尽。"""


def require(ok: bool, message: str) -> None:
    if not ok:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def bounded_text(value, limit: int, label: str = "text") -> str:
    require(type(value) is str, f"{label} 必须是字符串")
    try:
        size = len(value.encode("utf-8", errors="strict"))
    except UnicodeError as exc:
        raise LessonError(f"{label} 包含非法 Unicode") from exc
    require(size <= limit, f"{label} 超长")
    return value


def strict_json(text: str):
    bounded_text(text, MAX_RESPONSE_BYTES, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复 JSON 键")
            result[key] = value
        return result
    def invalid_constant(_):
        raise LessonError("不允许 NaN/Infinity")
    def small_int(raw):
        require(len(raw.lstrip("-")) <= 24, "JSON 整数过长")
        return int(raw)
    try:
        value = json.loads(text, object_pairs_hook=pairs,
                           parse_constant=invalid_constant, parse_int=small_int)
    except (json.JSONDecodeError, RecursionError, ValueError) as exc:
        if isinstance(exc, LessonError):
            raise
        raise LessonError("非法或过深 JSON") from exc
    stack = [(value, 0)]
    seen = 0
    while stack:
        node, depth = stack.pop()
        seen += 1
        require(depth <= 16 and seen <= 2048, "JSON 结构过大")
        if type(node) is float:
            require(math.isfinite(node), "JSON 非有限数值")
        elif type(node) is str:
            bounded_text(node, MAX_RESPONSE_BYTES, "JSON 字符串")
        elif type(node) is list:
            stack.extend((item, depth + 1) for item in node)
        elif type(node) is dict:
            stack.extend((item, depth + 1) for item in node.values())
    return value


def exact_keys(value, keys: set[str]) -> None:
    require(type(value) is dict and set(value) == keys, "字段集合不符合本题约定")


def parse_rational(value: str) -> Fraction:
    """整数、小数、分数的精确解析。不接受指数、百分号、多个答案或 float。"""
    text = bounded_text(value, 64, "answer").strip()
    require(NUMBER_RE.fullmatch(text) is not None, "需要一个有符号数值字符串")
    require(sum(c.isdigit() for c in text) <= 24, "数值有效位过多")
    try:
        result = Fraction(text)  # 不能先调用 float()
    except (ValueError, ZeroDivisionError) as exc:
        raise LessonError("无效分数或零分母") from exc
    return result


def close_rational(actual: str, expected: str, *, atol: str, rtol: str) -> bool:
    """独立的近似任务示例；不用在默认精确题上。规则：绝对项+相对项。"""
    a, e = parse_rational(actual), parse_rational(expected)
    absolute, relative = parse_rational(atol), parse_rational(rtol)
    require(absolute >= 0 and relative >= 0, "容差不能为负")
    return abs(a - e) <= absolute + relative * abs(e)

In [3]:
# (物理量类别, 换算成基准单位的精确倍数)
UNITS = {
    "s": ("time", "1"), "秒": ("time", "1"),
    "min": ("time", "60"), "minute": ("time", "60"), "分钟": ("time", "60"),
    "h": ("time", "3600"), "小时": ("time", "3600"),
    "m": ("length", "1"), "米": ("length", "1"),
    "cm": ("length", "1/100"), "厘米": ("length", "1/100"),
    "mm": ("length", "1/1000"),
    "CNY": ("money", "1"), "元": ("money", "1"), "fen": ("money", "1/100"),
}


def normalize_quantity(answer: str, unit: str) -> tuple[str, Fraction]:
    bounded_text(unit, 32, "unit")
    unit = unit.strip()
    require(unit in UNITS, "单位不在本课显式换算表中")
    dimension, scale = UNITS[unit]
    return dimension, parse_rational(answer) * Fraction(scale)


def verify_exact(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"answer"})
    actual, expected = parse_rational(payload["answer"]), Fraction(task["expected"])
    return {"passed": actual == expected, "canonical_answer": str(actual),
            "reason": "equal" if actual == expected else "wrong_value",
            "scope": "final_exact_value_not_reasoning_process"}


def verify_quantity(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"answer", "unit"})
    dim, val = normalize_quantity(payload["answer"], payload["unit"])
    target_dim, target_val = normalize_quantity(task["expected"], task["unit"])
    passed = dim == target_dim and val == target_val
    return {"passed": passed, "dimension": dim, "canonical_value": str(val),
            "reason": "equal_quantity" if passed else
                      ("wrong_dimension" if dim != target_dim else "wrong_quantity"),
            "scope": "quantity_with_explicit_unit_table"}


def verify_witness(task: dict, payload: dict) -> dict:
    """检查构造是否满足全部约束，不强求等于某一个参考解。"""
    exact_keys(payload, {"items"})
    items = payload["items"]
    require(type(items) is list and len(items) <= 16, "items 必须是有界列表")
    require(all(type(x) is int and abs(x) <= MAX_INTEGER for x in items),
            "items 只允许有界整数；bool 不算整数答案")
    checks = {
        "count": len(items) == task["count"],
        "membership": all(x in task["allowed"] for x in items),
        "distinct": len(set(items)) == len(items),
        "sum": sum(items) == task["target_sum"],
    }
    return {"passed": all(checks.values()), "checks": checks,
            "canonical_items": sorted(items), "scope": "all_stated_finite_constraints"}

In [4]:
OUTPUT_ROOT = Path("lesson_56_outputs")
print("本次工作目录：", Path.cwd())
print("只使用标准库；不会自动下载模型。")
print("1/2、0.5、2/4 →", [str(parse_rational(x)) for x in ["1/2", "0.5", "2/4"]])

本次工作目录： /mnt/data/.lesson56_build/notebook_run
只使用标准库；不会自动下载模型。
1/2、0.5、2/4 → ['1/2', '1/2', '1/2']


## 1. 多次采样与 Self-consistency：不要只看术语

原始 **Self-consistency** 方法先采样多条推理路径，再聚合它们的最终答案，不需要额外训练一个验证模型。[论文](https://arxiv.org/abs/2203.11171)

本课默认只接收最终 JSON，真实采样也关闭 Qwen thinking。因此，准确名称是**答案级规范化投票**，用来学习 Self-consistency 的聚合思想，**不是原论文的 CoT 实验复现**。我们不从结果推断模型内部推理路径的质量。

**Best-of-N** 则先为 N 个候选打分，再选择分数最高的一个。学习到的验证器是一种方式，也可以先用显式公开规则构建受限基线。[验证器论文](https://arxiv.org/abs/2110.14168)

三个选择器都只能选择已有候选，不能临时拼出一个新答案，更不能看参考答案后挑正确的那个。

### 1.1 沿用有限代码解释器，而不是执行任意模型 Python

以下实现来自前课的同一有限语言：只接受 `solve(numbers)`，提供有界整数、局部变量、比较、循环和返回；不允许导入、属性访问、任意函数调用、文件、网络或 Shell。

源码、AST、输入长度、整数范围和解释步骤都有限制。它不是通用 Python 沙箱，也不是程序正确性的形式化证明；即便通过额外测试，也只是对这些测试通过。

In [5]:
BIN_OPS = {ast.Add, ast.Sub, ast.Mult, ast.FloorDiv, ast.Mod}
CMP_OPS = {ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE}
ALLOWED_AST = {
    ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign, ast.AugAssign,
    ast.For, ast.If, ast.Return, ast.Name, ast.Load, ast.Store, ast.Constant,
    ast.BinOp, ast.UnaryOp, ast.UAdd, ast.USub, ast.Not, ast.Compare,
    ast.BoolOp, ast.And, ast.Or,
} | BIN_OPS | CMP_OPS


def int_value(value) -> int:
    require(type(value) is int and abs(value) <= MAX_INTEGER, "需要有界整数，不能使用bool")
    return value


def local_name(name: str) -> None:
    require(re.fullmatch(r"[a-z][a-z0-9_]{0,23}", name) is not None
            and "__" not in name and name not in {"solve"}, "非法局部变量名")


def parse_program(source: str) -> ast.FunctionDef:
    try:
        bounded_text(source, MAX_SOURCE_BYTES, "source")
        tree = ast.parse(source, mode="exec")  # 仅构建语法树，不编译执行
        require(len(tree.body) == 1 and type(tree.body[0]) is ast.FunctionDef,
                "只允许一个 solve(numbers) 函数")
        func = tree.body[0]
        args = func.args
        require(func.name == "solve" and not func.decorator_list and func.returns is None
                and not getattr(func, "type_params", []) and func.type_comment is None,
                "函数头不符合约定")
        require(len(args.args) == 1 and args.args[0].arg == "numbers"
                and args.args[0].annotation is None and not args.posonlyargs
                and not args.kwonlyargs and not args.defaults and not args.kw_defaults
                and args.vararg is None and args.kwarg is None, "参数必须只有 numbers")
        stack, total, names = [(tree, 0)], 0, set()
        while stack:
            node, depth = stack.pop()
            total += 1
            require(total <= MAX_AST_NODES and depth <= MAX_AST_DEPTH, "AST过大或过深")
            require(type(node) in ALLOWED_AST, f"不支持的语法：{type(node).__name__}")
            if isinstance(node, ast.FunctionDef):
                require(node is func, "不允许嵌套函数")
            if isinstance(node, ast.Constant):
                int_value(node.value)
            if isinstance(node, ast.Name):
                local_name(node.id)
                if isinstance(node.ctx, ast.Store):
                    require(node.id != "numbers", "不能改写输入")
                    names.add(node.id)
                    require(len(names) <= 16, "局部变量太多")
            if isinstance(node, ast.Assign):
                require(len(node.targets) == 1 and isinstance(node.targets[0], ast.Name),
                        "只支持一个局部变量赋值")
            if isinstance(node, ast.AugAssign):
                require(isinstance(node.target, ast.Name) and type(node.op) in BIN_OPS,
                        "不支持的增量赋值")
            if isinstance(node, ast.For):
                require(isinstance(node.target, ast.Name)
                        and isinstance(node.iter, ast.Name) and node.iter.id == "numbers"
                        and not node.orelse, "for 只能遍历原始 numbers，无 else")
            if isinstance(node, ast.Compare):
                require(len(node.ops) == len(node.comparators) == 1,
                        "比较必须是单一比较")
            if isinstance(node, ast.Return):
                require(node.value is not None, "需要返回整数")
            stack.extend((child, depth + 1) for child in ast.iter_child_nodes(node))
        return func
    except (SyntaxError, RecursionError, ValueError) as exc:
        raise UnsupportedCode(str(exc)[:180]) from exc


class _Returned(Exception):
    def __init__(self, value):
        self.value = int_value(value)


class TinyRunner:
    """解释整数语言。没有 eval/exec；有限输入与步骤不是通用OS沙箱。"""
    def __init__(self, program: ast.FunctionDef, step_limit: int = MAX_STEPS):
        require(type(step_limit) is int and 0 < step_limit <= 10000, "非法步骤上限")
        self.program, self.limit = program, step_limit
        self.used, self.env, self.numbers = 0, {}, ()

    def tick(self):
        self.used += 1
        if self.used > self.limit:
            raise RuntimeLimit("step_budget_exhausted")

    def binary(self, op, left, right):
        a, b = int_value(left), int_value(right)
        if isinstance(op, ast.Add): value = a + b
        elif isinstance(op, ast.Sub): value = a - b
        elif isinstance(op, ast.Mult): value = a * b
        elif isinstance(op, ast.FloorDiv):
            require(b != 0, "division_by_zero")
            value = a // b
        elif isinstance(op, ast.Mod):
            require(b != 0, "division_by_zero")
            value = a % b
        else: raise LessonError("unknown_operator")
        return int_value(value)

    def expr(self, node):
        self.tick()
        if isinstance(node, ast.Constant): return int_value(node.value)
        if isinstance(node, ast.Name):
            require(node.id != "numbers" and node.id in self.env, "uninitialized_or_non_scalar")
            return self.env[node.id]
        if isinstance(node, ast.BinOp):
            return self.binary(node.op, self.expr(node.left), self.expr(node.right))
        if isinstance(node, ast.UnaryOp):
            val = self.expr(node.operand)
            if isinstance(node.op, ast.Not):
                require(type(val) is bool, "not 仅用于布尔条件")
                return not val
            val = int_value(val)
            return val if isinstance(node.op, ast.UAdd) else -val
        if isinstance(node, ast.Compare):
            a, b = int_value(self.expr(node.left)), int_value(self.expr(node.comparators[0]))
            op = node.ops[0]
            if isinstance(op, ast.Eq): return a == b
            if isinstance(op, ast.NotEq): return a != b
            if isinstance(op, ast.Lt): return a < b
            if isinstance(op, ast.LtE): return a <= b
            if isinstance(op, ast.Gt): return a > b
            return a >= b
        if isinstance(node, ast.BoolOp):
            for child in node.values:
                val = self.expr(child)
                require(type(val) is bool, "and/or 仅用于布尔条件")
                if isinstance(node.op, ast.And) and not val: return False
                if isinstance(node.op, ast.Or) and val: return True
            return isinstance(node.op, ast.And)
        raise LessonError("unsupported_expression")

    def statements(self, body):
        for node in body:
            self.tick()
            if isinstance(node, ast.Assign):
                self.env[node.targets[0].id] = int_value(self.expr(node.value))
            elif isinstance(node, ast.AugAssign):
                name = node.target.id
                require(name in self.env, "uninitialized_variable")
                self.env[name] = self.binary(node.op, self.env[name], self.expr(node.value))
            elif isinstance(node, ast.If):
                cond = self.expr(node.test)
                require(type(cond) is bool, "if 需要比较条件")
                self.statements(node.body if cond else node.orelse)
            elif isinstance(node, ast.For):
                for item in self.numbers:
                    self.tick()
                    self.env[node.target.id] = item
                    self.statements(node.body)
            elif isinstance(node, ast.Return):
                raise _Returned(self.expr(node.value))
            else:
                raise LessonError("unsupported_statement")

    def run(self, numbers: list[int]) -> int:
        require(type(numbers) is list and len(numbers) <= MAX_INPUT_LEN, "输入必须是短整数列表")
        require(all(type(x) is int and abs(x) <= 1000 for x in numbers), "输入整数范围错误")
        self.used, self.env, self.numbers = 0, {}, tuple(numbers)
        try:
            self.statements(self.program.body)
        except _Returned as done:
            return done.value
        raise LessonError("missing_return")


def run_source(source: str, numbers: list[int], step_limit: int = MAX_STEPS) -> int:
    return TinyRunner(parse_program(source), step_limit).run(numbers)

In [6]:
CORRECT_CODE = {
    "sum_abs": """def solve(numbers):
    total = 0
    for value in numbers:
        if value < 0:
            total = total - value
        else:
            total = total + value
    return total
""",
    "count_nonzero": """def solve(numbers):
    count = 0
    for value in numbers:
        if value != 0:
            count = count + 1
    return count
""",
    "max_or_zero": """def solve(numbers):
    best = 0
    seen = 0
    for value in numbers:
        if seen == 0 or value > best:
            best = value
        seen = 1
    return best
""",
    "sum_nonnegative": """def solve(numbers):
    total = 0
    for value in numbers:
        if value >= 0:
            total = total + value
    return total
""",
}
WRONG_CODE = {
    "count_nonzero": CORRECT_CODE["count_nonzero"].replace("value != 0", "value > 0"),
    "max_or_zero": CORRECT_CODE["max_or_zero"].replace("seen == 0 or value > best", "value > best"),
    "sum_nonnegative": CORRECT_CODE["sum_nonnegative"].replace("value >= 0", "value >= 0 and value < 10"),
}
LANGUAGE_RULE = (
    "仅输出JSON {\"code\":\"...\"}。代码仅允许 def solve(numbers)，返回整数。"
    "输入是长度0至16、元素为[-1000,1000]整数的列表。只允许整数局部变量、"
    "+ - * // %、单一比较、and/or/not、if/else、for value in numbers、return；"
    "禁止函数调用(包括sum/len)、导入、属性、索引、while、递归、字符串、列表构造。"
    "源码最多4096个UTF-8字节；每例2000个解释步骤，整数绝对值不超过10^9。"
    "不要求解释推理，不使用代码围栏。"
)
NUMBER_RULE = (
    "仅输出JSON {\"answer\":\"数值\"}；数值是整数、普通十进制小数或分数的字符串，"
    "合计最多24个数字；保留正负号，不用指数、百分号、单位、解释或代码围栏。精确比较，不四舍五入。"
)
QUANTITY_RULE = (
    "仅输出JSON {\"answer\":\"数值\",\"unit\":\"单位\"}。数值用字符串；"
    "允许精确等价的单位换算。单位表：s/秒、min/minute/分钟、h/小时；"
    "m/米、cm/厘米、mm；CNY/元、fen(分)。不允许无单位，也不能混淆物理量。"
)
WITNESS_RULE = "仅输出JSON {\"items\":[整数,...]}。顺序不限，元素必须互不重复。不要输出理由。"


def trusted_oracle(name: str, numbers: list[int]) -> int:
    """宿主代码，不是执行候选。与候选解释器走不同计算路径。"""
    if name == "sum_abs": return sum(map(abs, numbers))
    if name == "count_nonzero": return len([x for x in numbers if x != 0])
    if name == "max_or_zero": return max(numbers, default=0)
    if name == "sum_nonnegative": return sum(x for x in numbers if x >= 0)
    raise LessonError("未知可信参考函数")


def code_cases(name: str) -> tuple[list[dict], list[dict]]:
    public_inputs = {
        "sum_abs": [[], [-1, 2], [0]],
        "count_nonzero": [[], [0, 0], [1, 0, 2]],
        "max_or_zero": [[], [1, 3, 2], [0]],
        "sum_nonnegative": [[], [1, 2], [-1, 2]],
    }[name]
    public = [{"input": xs, "expected": trusted_oracle(name, xs)} for xs in public_inputs]
    pool = [list(xs) for n in range(4) for xs in itertools.product([-2, -1, 0, 1, 2], repeat=n)]
    pool += [[10, -5, 20], [-12, -3], [1000, -1000, 0], [0] * 16,
             [-1] * 16, [10] * 16, [1000], [-1000]]
    seen = {tuple(xs) for xs in public_inputs}
    heldout = []
    for xs in pool:
        if tuple(xs) not in seen:
            seen.add(tuple(xs))
            heldout.append({"input": xs, "expected": trusted_oracle(name, xs)})
    return public, heldout


def make_tasks() -> list[dict]:
    tasks = []
    def add_numeric(id_, split, problem, expected):
        tasks.append({"id": id_, "split": split, "kind": "exact",
                      "problem": problem, "response_rule": NUMBER_RULE, "expected": expected})
    add_numeric("dev_fraction", "dev", "计算 3/4 + 1/8。", "7/8")
    add_numeric("fraction_sum", "test", "计算 1/3 + 1/6。", "1/2")
    add_numeric("signed_arithmetic", "test", "按通常运算顺序计算 -7 + 2 * 3。", "-1")
    add_numeric("decimal_sum", "test", "用精确十进制含义计算 0.1 + 0.2。", "3/10")
    for id_, split, problem, expected, unit in [
        ("dev_duration", "dev", "两小时等于多少分钟？", "120", "min"),
        ("duration", "test", "1.5小时等于多长时间？可用分钟作答。", "90", "min"),
        ("length", "test", "3/4米等于多长？可用厘米作答。", "75", "cm"),
        ("price", "test", "教学商店每杯茶2.50元，3杯总价多少？不计其他费用。", "7.5", "CNY"),
    ]:
        tasks.append({"id": id_, "split": split, "kind": "quantity", "problem": problem,
                      "expected": expected, "unit": unit, "response_rule": QUANTITY_RULE})
    for id_, split, allowed, count, total in [
        ("dev_subset", "dev", [1, 2, 3, 4, 5], 2, 6),
        ("subset_positive", "test", list(range(1, 9)), 3, 12),
        ("subset_signed", "test", [-3, -2, 0, 1, 4, 5], 2, 2),
        ("subset_primes", "test", [2, 3, 5, 7, 11, 13], 3, 21),
    ]:
        tasks.append({"id": id_, "split": split, "kind": "witness", "allowed": allowed,
                      "count": count, "target_sum": total,
                      "problem": f"从{allowed}选择{count}个互不相同的元素，使其和为{total}。给出任意合法选择。",
                      "response_rule": WITNESS_RULE})
    descriptions = {
        "sum_abs": "返回输入整数绝对值之和；空列表返回0。",
        "count_nonzero": "返回非零整数的数量；负数也算非零；空列表返回0。",
        "max_or_zero": "返回最大整数；只有空列表才返回0；全负列表不能错误返回0。",
        "sum_nonnegative": "返回所有非负整数之和；空列表返回0。",
    }
    for name, problem in descriptions.items():
        public, evaluation = code_cases(name)
        tasks.append({"id": "code_" + name, "split": "dev" if name == "sum_abs" else "test",
                      "kind": "code", "problem": problem, "oracle": name,
                      "public_cases": public, "evaluation_cases": evaluation,
                      "response_rule": LANGUAGE_RULE})
    return tasks

def run_cases(program: ast.FunctionDef, cases: list[dict]) -> dict:
    rows = []
    for case in cases:
        error, actual, used = None, None, 0
        runner = TinyRunner(program)
        try:
            actual = runner.run(copy.deepcopy(case["input"]))
        except LessonError as exc:
            error = str(exc)
        used = runner.used
        rows.append({"input": case["input"], "expected": case["expected"],
                     "actual": actual, "error": error, "steps": used,
                     "passed": error is None and type(actual) is int and actual == case["expected"]})
    return {"passed": all(row["passed"] for row in rows),
            "passed_count": sum(row["passed"] for row in rows), "total": len(rows), "cases": rows}

def verify_code(task: dict, payload: dict) -> dict:
    exact_keys(payload, {"code"})
    program = parse_program(payload["code"])
    public = run_cases(program, task["public_cases"])
    evaluation = run_cases(program, task["evaluation_cases"])
    return {"passed": public["passed"] and evaluation["passed"],
            "public": public, "evaluation": evaluation,
            "scope": "finite_tests_not_all_inputs_not_code_security",
            "source_sha256": hashlib.sha256(payload["code"].encode()).hexdigest()}


# The dataset and bounded graders above preserve lesson 55's protocol.
# They are embedded for standalone use. Reference answers below belong only to grading.
SOURCES.update({
    "self_consistency": "https://arxiv.org/abs/2203.11171",
    "best_of_n": "https://arxiv.org/abs/2110.14168",
    "pass_at_k": "https://arxiv.org/abs/2107.03374",
    "human_eval": "https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py",
    "test_time_compute": "https://arxiv.org/abs/2408.03314",
    "generation": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation",
    "randomness": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
})
MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"

## 2. 先登记候选位置，再开始生成

默认 N=5，每题计划槽为 `s0…s4`。四类测试题各 3 道，因此共有 **60 个计划候选槽**。

缺失、错误 JSON、超预算和运行异常都占用原来的槽，不自动补抽。第一候选基线固定取 `s0`，不能悄悄改成“第一条正确完成的响应”。

随机种子根据 `(基础种子, task_id, sample_index)` 派生，不依赖遍历顺序。每次反复重置同一个 seed，可能只是在重复同一个样本。

> `first` 是同一采样分布中的第一条，**不是贪心解码基线**。贪心需要另一份事先登记的实验，不能混用名称。

In [7]:
def public_view(task: dict) -> dict:
    """Whitelist only information available before inference. No gold values or heldout tests."""
    view = {"task_id": task["id"], "kind": task["kind"],
            "problem": task["problem"], "response_rule": task["response_rule"]}
    if task["kind"] == "witness":
        view["constraints"] = {k: copy.deepcopy(task[k]) for k in ("allowed", "count", "target_sum")}
    if task["kind"] == "code":
        view["public_examples"] = copy.deepcopy(task["public_cases"])
    return view


def validate_view(view: dict) -> None:
    require(type(view) is dict, "公共视图必须是字典")
    kind = view.get("kind")
    require(kind in {"exact", "quantity", "witness", "code"}, "未知任务类型")
    keys = {"task_id", "kind", "problem", "response_rule"}
    if kind == "witness": keys.add("constraints")
    if kind == "code": keys.add("public_examples")
    exact_keys(view, keys)
    for k in ("task_id", "problem", "response_rule"):
        bounded_text(view[k], 8192, k)
    if kind == "witness":
        exact_keys(view["constraints"], {"allowed", "count", "target_sum"})


def stable_seed(task_id: str, sample_index: int, base_seed: int = 5600) -> int:
    require(type(sample_index) is int and 0 <= sample_index < 32, "样本索引越界")
    require(type(base_seed) is int and base_seed >= 0, "随机种子应为非负整数")
    raw = canonical([VERSION, base_seed, task_id, sample_index]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], "big") % (2**31)


def build_plan(split: str = "test", n: int = 5, base_seed: int = 5600) -> dict:
    require(split in {"dev", "test"}, "split必须是dev或test")
    require(type(n) is int and 1 <= n <= 16, "候选数应为1至16的整数")
    tasks = [t for t in make_tasks() if t["split"] == split]
    requests = [public_view(t) for t in tasks]
    plan = {
        "version": VERSION, "split": split, "n": n, "base_seed": base_seed,
        "task_ids": [t["id"] for t in tasks], "dataset_sha256": digest(tasks),
        "public_requests_sha256": digest(requests),
        "selectors": ["first", "canonical_vote", "public_best_of_n"],
        "selection_contract": {
            "tie": "earliest_planned_index", "failure": "retain_planned_slot",
            "numeric_proxy": "format_only_not_correctness",
            "witness_proxy": "fraction_of_public_constraints_satisfied",
            "code_proxy": "fraction_of_public_examples_passed",
            "code_vote": "same_outputs_on_public_examples_not_semantic_equivalence",
            "completion": "only_ok_and_eos_are_eligible",
        },
        "sampling": {"do_sample": True, "temperature": 0.8, "top_p": 0.95,
                     "top_k": 20, "max_new_tokens": 256, "enable_thinking": False,
                     "context_limit": 2048, "sequential": True,
                     "mode": "answer_only_not_paper_CoT_replication"},
        "slots": [{"task_id": t["id"], "sample_id": f"s{i}",
                   "seed": stable_seed(t["id"], i, base_seed),
                   "request_sha256": digest(public_view(t))}
                  for t in tasks for i in range(n)],
    }
    return plan


def validate_plan(plan: dict) -> None:
    require(type(plan) is dict and plan.get("version") == VERSION, "计划版本不匹配")
    expected = build_plan(plan.get("split"), plan.get("n"), plan.get("base_seed"))
    require(plan == expected, "计划、题集或采样/选择协议被修改；请建立新实验版本")


def make_record(task_id: str, i: int, payload=None, *, raw: str | None = None,
                status: str = "ok", stop: str = "eos", base_seed: int = 5600,
                usage: dict | None = None) -> dict:
    return {"task_id": task_id, "sample_id": f"s{i}",
            "seed": stable_seed(task_id, i, base_seed), "status": status,
            "stop_reason": stop,
            "output": raw if raw is not None else canonical(payload),
            "usage": usage or {"input_tokens": None, "output_tokens": None,
                               "latency_ms": None}}


def validate_record(record: dict, slot: dict) -> None:
    exact_keys(record, {"task_id", "sample_id", "seed", "status", "stop_reason", "output", "usage"})
    for key in ("task_id", "sample_id", "seed"):
        require(type(record[key]) is type(slot[key]) and record[key] == slot[key], "候选身份或seed不符")
    require(record["status"] in {"ok", "error"}, "status非法")
    require(record["stop_reason"] in {"eos", "length", "error"}, "stop_reason非法")
    require((record["status"] == "error") == (record["stop_reason"] == "error"), "状态与停止原因不符")
    bounded_text(record["output"], MAX_RESPONSE_BYTES, "output")
    exact_keys(record["usage"], {"input_tokens", "output_tokens", "latency_ms"})
    for key in ("input_tokens", "output_tokens"):
        value = record["usage"][key]
        require(value is None or (type(value) is int and 0 <= value <= 10**7), "token计数非法")
    ms = record["usage"]["latency_ms"]
    require(ms is None or (type(ms) in {int, float} and math.isfinite(ms) and ms >= 0), "计时非法")


def index_records(plan: dict, records: list[dict]) -> dict:
    validate_plan(plan)
    require(type(records) is list and len(records) <= len(plan["slots"]), "响应数量越界")
    slots = {(s["task_id"], s["sample_id"]): s for s in plan["slots"]}
    indexed = {}
    for record in records:
        require(type(record) is dict, "候选记录必须是字典")
        require(type(record.get("task_id")) is str and type(record.get("sample_id")) is str, "候选ID必须是字符串")
        key = (record.get("task_id"), record.get("sample_id"))
        require(key in slots, "未知题目或样本ID")
        require(key not in indexed, "重复样本槽，不能当作额外一票")
        validate_record(record, slots[key])
        indexed[key] = copy.deepcopy(record)
    return indexed

In [8]:
PLAN = build_plan("test", n=5)
print("题目数：", len(PLAN["task_ids"]))
print("计划候选数：", len(PLAN["slots"]))
print("前3个样本槽：")
for slot in PLAN["slots"][:3]:
    print(slot["task_id"], slot["sample_id"], slot["seed"])
task_map = {t["id"]: t for t in make_tasks()}
print("数值题公开字段：", sorted(public_view(task_map["fraction_sum"])))
print("代码题公开字段：", sorted(public_view(task_map["code_count_nonzero"])))

题目数： 12
计划候选数： 60
前3个样本槽：
fraction_sum s0 2060571765
fraction_sum s1 457901564
fraction_sum s2 1041143082
数值题公开字段： ['kind', 'problem', 'response_rule', 'task_id']
代码题公开字段： ['kind', 'problem', 'public_examples', 'response_rule', 'task_id']


## 3. 投票前先定义“同一个答案”

直接对字符串投票会把 `"1/2"`、`"0.5"`、`"2/4"` 分成三组。本课通过 `Fraction` 从原始字符串得到精确值，不先经过 `float`。[Python 文档](https://docs.python.org/3.13/library/fractions.html)

| 类型 | 本课规范化 |
|---|---|
| 精确数值 | 最简有理数，保留正负号 |
| 物理量 | 物理量类别＋基准单位下的精确值 |
| 约束见证 | 整数列表排序，但不删除重复元素 |
| 代码 | 对相同公开输入的行为签名，不宣称语义等价 |

代码的不同实现可能在公开样例上完全一致，但在其他输入上不同。多解任务中的不同合法见证也可能分票，因此不能把“越一致”当成适用于一切任务的目标。

本课使用**相对多数**，不是强制超过 50%。票数相同时固定选最早计划位置。格式不合法或未完整生成的响应不投票，但保留在计划分母和成本中。

In [9]:
def observe_candidate(view: dict, record: dict | None, sample_index: int) -> dict:
    """Selection input: public task + raw response only. Never access expected/evaluation_cases."""
    validate_view(view)
    item = {"index": sample_index, "sample_id": f"s{sample_index}", "eligible": False,
            "key": None, "proxy_score": None, "reason": "missing",
            "public_code_tests": 0, "public_constraint_checks": 0}
    if record is None:
        return item
    if record["status"] != "ok":
        item["reason"] = "generation_error"
        return item
    if record["stop_reason"] != "eos":
        item["reason"] = "incomplete_generation"
        return item  # Do not execute partial code or quietly repair its JSON.
    try:
        payload = strict_json(record["output"])
        kind = view["kind"]
        if kind == "exact":
            exact_keys(payload, {"answer"})
            key = canonical(["rational", str(parse_rational(payload["answer"]))])
            score = 1.0  # A format check is not a mathematical verifier.
            reason = "format_only"
        elif kind == "quantity":
            exact_keys(payload, {"answer", "unit"})
            dim, value = normalize_quantity(payload["answer"], payload["unit"])
            key, score = canonical(["quantity", dim, str(value)]), 1.0
            reason = "format_and_known_unit_only"
        elif kind == "witness":
            exact_keys(payload, {"items"})
            items = payload["items"]
            require(type(items) is list and len(items) <= 16, "items需为有界列表")
            require(all(type(x) is int and abs(x) <= MAX_INTEGER for x in items), "见证元素类型非法")
            checks = verify_witness(view["constraints"], payload)["checks"]
            key = canonical(["witness_set_order_free", sorted(items)])
            score = sum(checks.values()) / len(checks)
            item["public_constraint_checks"] = len(checks)
            reason = "public_constraints_not_hidden_reference"
        else:
            exact_keys(payload, {"code"})
            result = run_cases(parse_program(payload["code"]), view["public_examples"])
            key = canonical(["public_behavior", [[r["error"], r["actual"]] for r in result["cases"]]])
            score = result["passed_count"] / result["total"]
            item["public_code_tests"] = result["total"]
            reason = "public_examples_only_not_functional_proof"
        item.update(eligible=True, key=key, proxy_score=score, reason=reason)
    except LessonError as exc:
        item["reason"] = "invalid_or_unsupported: " + str(exc)
    return item


def select_first(observations: list[dict]) -> int | None:
    """First planned slot, not the first successful candidate; no resampling after failure."""
    require(bool(observations), "没有计划候选")
    return observations[0]["index"]


def select_vote(observations: list[dict]) -> dict:
    """Plurality of canonical answers; tie -> earliest planned occurrence. No deduplication."""
    require(bool(observations), "没有计划候选")
    groups = {}
    for o in observations:
        if o["eligible"]:
            groups.setdefault(o["key"], []).append(o["index"])
    if not groups:
        return {"selected_index": None, "votes": 0, "valid": 0,
                "agreement_valid": None, "agreement_planned": 0.0,
                "tie": False, "groups": {}}
    ordered = sorted(groups, key=lambda k: (-len(groups[k]), min(groups[k])))
    winner = ordered[0]
    votes, valid = len(groups[winner]), sum(map(len, groups.values()))
    return {"selected_index": min(groups[winner]), "votes": votes, "valid": valid,
            "agreement_valid": votes / valid, "agreement_planned": votes / len(observations),
            "tie": len(ordered) > 1 and len(groups[ordered[1]]) == votes,
            "groups": {k: groups[k] for k in ordered}}


def select_best(observations: list[dict]) -> int | None:
    eligible = [o for o in observations if o["eligible"]]
    if not eligible: return None
    for o in eligible:
        require(type(o["proxy_score"]) in {int, float} and math.isfinite(o["proxy_score"]), "非法代理分数")
    return min(eligible, key=lambda o: (-o["proxy_score"], o["index"]))["index"]


def vote_locked(observations: list[dict], remaining: int) -> bool:
    """Only a fixed-N vote cannot be overturned; does not prove correctness or statistical confidence."""
    require(type(remaining) is int and remaining >= 0, "remaining必须是非负整数")
    counts = sorted(Counter(o["key"] for o in observations if o["eligible"]).values(), reverse=True)
    if not counts: return False
    first = counts[0]
    second = counts[1] if len(counts) > 1 else 0
    return first > second + remaining


def replay_early_vote(observations: list[dict]) -> dict:
    total = len(observations)
    require(total > 0, "没有计划槽")
    for consumed in range(1, total + 1):
        prefix = observations[:consumed]
        if vote_locked(prefix, total - consumed) or consumed == total:
            result = select_vote(prefix)
            return {"consumed_slots": consumed, "potential_unrequested_slots": total - consumed,
                    "selected_index": result["selected_index"],
                    "label": "offline_prefix_replay_not_measured_generation_savings"}
    raise AssertionError("unreachable")

In [10]:
fraction_view = public_view(task_map["fraction_sum"])
answers = ["1/2", "0.5", "2/4", "3/5", "0.6"]
observed = [observe_candidate(fraction_view,
    make_record("fraction_sum", i, {"answer": answer}), i)
    for i, answer in enumerate(answers)]
vote_result = select_vote(observed)
print("原字符串：", answers)
print("规范化结果：", [o["key"] for o in observed])
print("选中索引：", vote_result["selected_index"], "票数：", vote_result["votes"])

原字符串： ['1/2', '0.5', '2/4', '3/5', '0.6']
规范化结果： ['["rational","1/2"]', '["rational","1/2"]', '["rational","1/2"]', '["rational","3/5"]', '["rational","3/5"]']
选中索引： 0 票数： 3


### 3.1 高一致率可能只是“一起答错”

下面候选的正确值是 1/2，但前三条都答成 3/5。增加采样可能让一个系统性错误形成更稳定的多数。

还要分别记录：

\[
\text{有效票一致率}=\frac{\text{获胜票数}}{\text{有效候选数}}
\]

\[
\text{计划槽支持率}=\frac{\text{获胜票数}}{\text{原计划候选数}}
\]

一条有效、四条缺失，可以得到 100% 的有效票一致率，但原计划槽支持率只有 20%。**两者都不是校准后的正确概率。**

In [11]:
wrong_answers = ["3/5", "3/5", "3/5", "1/2", "1/2"]
wrong_obs = [observe_candidate(fraction_view, make_record("fraction_sum", i, {"answer": x}), i)
             for i, x in enumerate(wrong_answers)]
print("错误多数：", select_vote(wrong_obs)["selected_index"], select_vote(wrong_obs)["votes"])
one_valid = [observed[0]] + [observe_candidate(fraction_view, None, i) for i in range(1, 5)]
print("只有一条有效：", {k: select_vote(one_valid)[k] for k in ["agreement_valid", "agreement_planned"]})

错误多数： 0 3
只有一条有效： {'agreement_valid': 1.0, 'agreement_planned': 0.2}


### 3.2 多个不同见证都可以正确

从 1 到 8 选择三个不同整数，总和为 12。`[1,4,7]`、`[2,4,6]`、`[3,4,5]` 都合法，投票却把它们分成三组。

相反，错误的 `[1,2,3]` 重复出现两次，可以赢过每个只出现一次的合法解。

对这种任务，检查公开约束可能比统计字符串的一致性更合适。选择器使用的是题目已经公开的约束，不是偷偷获取某一份标准答案。

In [12]:
witness_view = public_view(task_map["subset_positive"])
values = [[1,2,3], [1,4,7], [2,4,6], [3,4,5], [1,2,3]]
wo = [observe_candidate(witness_view, make_record("subset_positive", i, {"items": x}), i)
      for i,x in enumerate(values)]
print("投票选择：", select_vote(wo)["selected_index"])
print("公开约束评分：", [o["proxy_score"] for o in wo])
print("Best-of-N 选择：", select_best(wo))

投票选择： 0
公开约束评分： [0.75, 1.0, 1.0, 1.0, 0.75]
Best-of-N 选择： 1


## 4. Best-of-N：选择器到底知道什么？

\[
\widehat y=\arg\max_{y_i\in\{y_1,\dots,y_N\}}s(q,y_i)
\]

本课不是训练奖励模型，而是建立一个**公开规则代理分数**：

| 类型 | 代理分数 | 局限 |
|---|---|---|
| 数值／单位 | 格式合法得 1 分 | 没有验证答案是否正确，故意保留弱基线 |
| 见证 | 四条公共约束通过的比例 | 对该有限约束有意义，不是通用推理分数 |
| 代码 | 公开样例通过比例 | 可能被只适配样例的错误实现骗过 |

分数并列时取最早计划索引，全部无效则不选。不能把候选自己的 `score` 或“我确信正确”直接当作可信分数。

本课不让数值代理读取 `expected`，不让代码代理读取 `evaluation_cases`。留出目标仅出现在下一阶段。

In [13]:
def finalize_selection(plan: dict, records: list[dict]) -> dict:
    indexed = index_records(plan, records)
    views = {t["id"]: public_view(t) for t in make_tasks() if t["split"] == plan["split"]}
    rows = []
    for task_id in plan["task_ids"]:
        observations = [observe_candidate(views[task_id], indexed.get((task_id, f"s{i}")), i)
                        for i in range(plan["n"])]
        vote = select_vote(observations)
        selected = {"first": select_first(observations),
                    "canonical_vote": vote["selected_index"],
                    "public_best_of_n": select_best(observations)}
        rows.append({"task_id": task_id, "kind": views[task_id]["kind"],
                     "observations": observations, "vote": vote,
                     "selected": selected, "early_vote": replay_early_vote(observations)})
    return {"version": VERSION, "plan_sha256": digest(plan),
            "records_sha256": digest(sorted(records, key=lambda r: (r["task_id"], r["sample_id"]))),
            "rows": rows, "gold_used": False,
            "scope": "public-only deterministic selector; not an OS security boundary"}


def grade_candidate(task: dict, record: dict | None) -> dict:
    row = {"strict_success": False, "stage": "missing"}
    if record is None: return row
    if record["status"] != "ok": return {**row, "stage": "generation_error"}
    if record["stop_reason"] != "eos": return {**row, "stage": "incomplete_generation"}
    try:
        payload = strict_json(record["output"])
        fn = {"exact": verify_exact, "quantity": verify_quantity,
              "witness": verify_witness, "code": verify_code}[task["kind"]]
        detail = fn(task, payload)
        row.update(strict_success=detail["passed"], stage="passed" if detail["passed"] else "wrong_result")
        if task["kind"] == "code":
            row["tests"] = {"public": [detail["public"]["passed_count"], detail["public"]["total"]],
                            "heldout": [detail["evaluation"]["passed_count"], detail["evaluation"]["total"]]}
    except UnsupportedCode:
        row["stage"] = "unsupported_code"
    except LessonError:
        row["stage"] = "invalid_format"
    return row


def estimate_pass_at_k(n: int, c: int, k: int) -> float:
    require(all(type(x) is int for x in (n, c, k)), "n,c,k必须是整数，不接受bool")
    require(1 <= k <= n <= 10000 and 0 <= c <= n, "pass@k参数越界")
    return 1.0 if n - c < k else 1.0 - math.comb(n-c, k) / math.comb(n, k)


def score_selection(plan: dict, records: list[dict], selection: dict) -> dict:
    """Only this phase receives private targets. Never feeds them back into selection."""
    indexed = index_records(plan, records)
    require(selection == finalize_selection(plan, records), "选择记录与公共协议或候选不一致")
    tasks = {t["id"]: t for t in make_tasks() if t["split"] == plan["split"]}
    rows = []
    for chosen in selection["rows"]:
        task_id = chosen["task_id"]
        grades = [grade_candidate(tasks[task_id], indexed.get((task_id, f"s{i}"))) for i in range(plan["n"])]
        successes = {s: (False if i is None else grades[i]["strict_success"])
                     for s, i in chosen["selected"].items()}
        any_correct = any(g["strict_success"] for g in grades)
        rows.append({"task_id": task_id, "kind": tasks[task_id]["kind"], "grades": grades,
                     "correct_candidates": sum(g["strict_success"] for g in grades),
                     "any_correct": any_correct, "selected_success": successes,
                     "selection_miss": {s: any_correct and not v for s, v in successes.items()}})
    count = len(rows)
    covered = sum(r["any_correct"] for r in rows)
    methods = {}
    for s in plan["selectors"]:
        correct = sum(r["selected_success"][s] for r in rows)
        by_kind = {}
        for kind in ("exact", "quantity", "witness", "code"):
            group = [r for r in rows if r["kind"] == kind]
            by_kind[kind] = {"correct": sum(r["selected_success"][s] for r in group), "planned": len(group)}
        methods[s] = {"correct": correct, "planned": count, "success_rate": correct / count,
                      "selection_misses": sum(r["selection_miss"][s] for r in rows),
                      "selection_efficiency_given_coverage": correct / covered if covered else None,
                      "by_kind": by_kind}
        require(correct <= covered, "选择器不应高于同候选池的覆盖上界")
    usable = [r for r in records if r["status"] == "ok" and r["stop_reason"] == "eos"]
    known = [r["usage"]["output_tokens"] for r in records if r["usage"]["output_tokens"] is not None]
    planned = len(plan["slots"])
    code_tests = sum(o["public_code_tests"] for row in selection["rows"] for o in row["observations"])
    constraints = sum(o["public_constraint_checks"] for row in selection["rows"] for o in row["observations"])
    return {"version": VERSION, "selection_sha256": digest(selection), "n": plan["n"],
            "coverage": {"correct": covered, "planned": count, "rate": covered/count,
                         "label": "observed_any_correct_not_deployable_or_iid_estimate"},
            "methods": methods, "rows": rows,
            "accounting": {"planned_candidate_slots": planned, "received_records": len(records),
                           "missing_slots": planned-len(records), "complete_generation_records": len(usable),
                           "known_output_tokens_sum": sum(known),
                           "unknown_output_token_slots": planned-len(known),
                           "all_slot_output_tokens": sum(known) if len(known) == planned else None,
                           "public_code_cases_per_selection_pass": code_tests,
                           "public_witness_conditions_per_selection_pass": constraints,
                           "selection_and_revalidation_passes": 2,
                           "public_selection_case_evaluations_two_passes": 2 * code_tests,
                           "private_grading_public_case_evaluations": sum(
                               g.get("tests", {}).get("public", [0,0])[1] for row in rows for g in row["grades"]),
                           "private_grading_heldout_case_evaluations": sum(
                               g.get("tests", {}).get("heldout", [0,0])[1] for row in rows for g in row["grades"]),
                           "work_scope": "单次write_evaluation：公共选择及重校验分别计数；不含run_checks、预算曲线或其他重评分",
                           "judge_model_calls": 0, "synthetic_model_latency_ms": None},
            "limitations": ["公开题库的有限测试，不是通用代码证明或安全沙箱。",
                            "覆盖率是评分后诊断上界，不能作为部署选择策略。",
                            "这里的数值代理只有格式检查，不是数学正确性验证器。"]}

In [14]:
code_task = task_map["code_count_nonzero"]
code_records = [
    make_record(code_task["id"], 0, {"code": WRONG_CODE["count_nonzero"]}),
    make_record(code_task["id"], 1, {"code": CORRECT_CODE["count_nonzero"]}),
]
code_obs = [observe_candidate(public_view(code_task), r, i) for i,r in enumerate(code_records)]
selected_before_grading = select_best(code_obs)
print("公开样例评分：", [o["proxy_score"] for o in code_obs])
print("先锁定选择：", selected_before_grading)
print("之后才用额外测试核对：")
for r in code_records:
    print(r["sample_id"], grade_candidate(code_task, r))

公开样例评分： [1.0, 1.0]
先锁定选择： 0
之后才用额外测试核对：
s0 {'strict_success': False, 'stage': 'wrong_result', 'tests': {'public': [3, 3], 'heldout': [40, 161]}}
s1 {'strict_success': True, 'stage': 'passed', 'tests': {'public': [3, 3], 'heldout': [161, 161]}}


### 4.1 更多候选不保证选择结果单调变好

同一个固定排序函数，如果对一个错误候选打出更高分，加入它之后，Best-of-N 就可能从正确答案切换成错误答案。

真正保证不下降的是**嵌套候选池里“是否至少有一个正确候选”**，不是代理选择后的质量。

选择器偏差、生成错误的相关性和预算分配都需要实际评测。测试时计算研究也区分了不同题目难度和计算分配方式，并没有宣称任意增加样本数都等效。[相关研究](https://arxiv.org/abs/2408.03314)

In [15]:
hand_scores = [
    {"index": 0, "eligible": True, "proxy_score": 0.80},  # 事后知道正确
    {"index": 1, "eligible": True, "proxy_score": 0.99},  # 事后知道错误
]
print("N=1选中：", select_best(hand_scores[:1]))
print("N=2选中：", select_best(hand_scores))
print("这两项分数是手工反例，不是学习到的验证器输出。")

N=1选中： 0
N=2选中： 1
这两项分数是手工反例，不是学习到的验证器输出。


## 5. 冻结候选池，公平地比较三种选择器

现在使用第 55 课同一套 12 道评测题，四类各 3 道；每题预先计划 5 个候选，全部选择器使用同一池。

本课手工记录刻意包含：等价表述、错误多数、不同合法见证、只过公开测试的代码、非法 JSON、截断、运行错误和一个缺失位置。**这不是生成数据，也不是任何模型的成绩。**

程序先写入 `selection.json`，然后才调用最终评分器。重新评分会验证原选择是否符合公共规则，不会利用最终目标反选最优项。

In [16]:
def fixture_records(plan: dict) -> list[dict]:
    require(plan["split"] == "test" and plan["n"] <= 5, "手工夹具仅支持test且N<=5")
    ans = lambda x: {"answer": x}
    qty = lambda x, u: {"answer": x, "unit": u}
    wit = lambda xs: {"items": xs}
    items = {
        "fraction_sum": [ans("2/5"), ans("1/2"), ans("0.5"), ans("2/4"), None],
        "signed_arithmetic": [ans("-1"), ans("1"), ans("1"), ans("1"), ans("2")],
        "decimal_sum": [ans("0.30000000000000004"), ans("3/10"), ans("0.3"), ans("0.30"), ans("3/10")],
        "duration": [qty("90","m"), qty("1.5","h"), qty("90","min"), qty("5400","s"), qty("90","min")],
        "length": [qty("75","cm"), qty("750","cm"), qty("7.5","m"), qty("7500","mm"), qty("3/4","m")],
        "price": [None, qty("7.5","CNY"), qty("750","fen"), None, None],
        "subset_positive": [wit([1,2,3]), wit([1,4,7]), wit([2,4,6]), wit([3,4,5]), wit([1,2,3])],
        "subset_signed": [wit([-3,5]), wit([1,1]), wit([1,1]), wit([1,1]), wit([-2,4])],
        "subset_primes": [None, wit([3,5,13]), wit([13,3,5]), wit([3,7,11]), None],
    }
    for name in ("count_nonzero", "max_or_zero", "sum_nonnegative"):
        items["code_"+name] = [
            {"code": WRONG_CODE[name]}, {"code": CORRECT_CODE[name]},
            {"code": WRONG_CODE[name]}, {"code": CORRECT_CODE[name]+"# alternate sample\n"},
            {"code": "def solve(numbers):\n    return len(numbers)\n"}]
    rows = []
    for task_id in plan["task_ids"]:
        for i, payload in enumerate(items[task_id][:plan["n"]]):
            if task_id == "subset_primes" and i == 4: continue  # Deliberately missing slot.
            kw = {}
            if payload is None:
                if (task_id,i) in {("fraction_sum",4), ("price",4)}:
                    kw = {"raw": "", "status": "error", "stop": "error"}
                else:
                    kw = {"raw": '{"answer":"7.5","answer":"75","unit":"CNY"}' if i==3 else "不是规定的JSON"}
            if task_id == "duration" and i == 4: kw["stop"] = "length"
            rows.append(make_record(task_id,i,payload,base_seed=plan["base_seed"],**kw))
    return rows


def mechanism_experiments() -> dict:
    task = next(t for t in make_tasks() if t["id"] == "fraction_sum")
    view = public_view(task)
    raws = ["1/2", "0.5", "2/4", "3/5", "0.6"]
    obs = [observe_candidate(view, make_record(task["id"],i,{"answer":x}), i) for i,x in enumerate(raws)]
    normalized = select_vote(obs)
    wrong_records = [make_record(task["id"],i,{"answer":x}) for i,x in enumerate(["3/5"]*3+["1/2"]*2)]
    wrong_obs = [observe_candidate(view,r,i) for i,r in enumerate(wrong_records)]
    leading = select_vote(wrong_obs)
    valid_one = [observe_candidate(view,make_record(task["id"],0,{"answer":"1/2"}),0)]
    valid_one += [observe_candidate(view,None,i) for i in range(1,5)]
    ct = next(t for t in make_tasks() if t["id"]=="code_count_nonzero")
    code_pool = [make_record(ct["id"],0,{"code":WRONG_CODE["count_nonzero"]}),
                 make_record(ct["id"],1,{"code":CORRECT_CODE["count_nonzero"]})]
    co = [observe_candidate(public_view(ct),r,i) for i,r in enumerate(code_pool)]
    # A false-positive score can become the maximum once a worse candidate is added.
    proxy = [{"index":0,"eligible":True,"proxy_score":0.8},
             {"index":1,"eligible":True,"proxy_score":0.99}]
    cost = {"N":8, "prompt_tokens_per_call":100, "output_tokens_per_call":40,
            "verifier_input_tokens_per_call":160, "verifier_output_tokens_per_call":10}
    return {"numeric_normalization": {"raw_answers":raws,"vote":normalized},
            "confidently_wrong": {"vote":leading,"correct":False,"early":replay_early_vote(wrong_obs)},
            "valid_only_denominator":select_vote(valid_one),
            "public_test_blindspot": {"proxy_scores":[o["proxy_score"] for o in co],
                                      "selected_index":select_best(co),
                                      "heldout_grades":[grade_candidate(ct,r) for r in code_pool]},
            "nonmonotonic_best_of_n": {"N1_selected_correct":select_best(proxy[:1])==0,
                                        "N2_selected_correct":select_best(proxy)==0,
                                        "label":"手工代理分数反例，不是学习到的验证器"},
            "pass_example":{"n":4,"c":2,"k":2,"estimate":estimate_pass_at_k(4,2,2)},
            "independence_example":{"p":0.6,"n":5,"iid_any":1-(1-0.6)**5,
                                    "identical_events_any":0.6,
                                    "assumption":"解析概率示例，非模型测量"},
            "hypothetical_cost":{**cost,"generation_tokens":8*(100+40),
                                 "verifier_tokens":8*(160+10),
                                 "total_tokens":8*(100+40+160+10),
                                 "label":"假设token数，不是实测且token总数不等于FLOPs"}}

In [17]:
def save_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)+"\n", encoding="utf-8")


def save_jsonl(path: Path, rows: list[dict]) -> None:
    path.write_text("".join(canonical(row)+"\n" for row in rows), encoding="utf-8")


def parse_document(text: str):
    """Bounded file JSON. Model outputs still use the stricter 16-KiB parser above."""
    bounded_text(text, MAX_FILE_BYTES, "文件")
    def pairs(items):
        out = {}
        for k,v in items:
            require(k not in out, "文件包含重复JSON键")
            out[k] = v
        return out
    def constant(_): raise LessonError("文件禁止非有限数")
    try:
        value = json.loads(text, object_pairs_hook=pairs, parse_constant=constant)
    except (ValueError, RecursionError) as exc:
        if isinstance(exc, LessonError): raise
        raise LessonError("文件JSON非法") from exc
    stack = [(value,0)]; count = 0
    while stack:
        x, depth = stack.pop(); count += 1
        require(depth <= 40 and count <= 200000, "文件结构预算超限")
        if type(x) is float: require(math.isfinite(x), "文件非有限数")
        if type(x) is dict: stack.extend((v,depth+1) for v in x.values())
        if type(x) is list: stack.extend((v,depth+1) for v in x)
    return value


def load_json(path: Path):
    require(path.is_file() and path.stat().st_size <= MAX_FILE_BYTES, "文件不存在或超大")
    return parse_document(path.read_text(encoding="utf-8"))


def load_jsonl(path: Path) -> list[dict]:
    require(path.is_file() and path.stat().st_size <= MAX_FILE_BYTES, "响应文件不存在或超大")
    rows = [parse_document(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    require(len(rows) <= MAX_RECORDS, "响应记录过多")
    return rows


def new_run(root: str | Path, label: str) -> Path:
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    return Path(tempfile.mkdtemp(prefix=f"{label}_{stamp}_", dir=root))


def messages_for(view: dict) -> list[dict]:
    validate_view(view)
    text = view["problem"]+"\n"+view["response_rule"]
    if "public_examples" in view:
        text += "\n公开样例：" + canonical(view["public_examples"])
    return [{"role":"system","content":"遵守题目给定的输出格式。只返回最终JSON，不附加解释或代码围栏。"},
            {"role":"user","content":text}]


def export_requests(plan: dict, out: Path) -> None:
    validate_plan(plan)
    views = {t["id"]:public_view(t) for t in make_tasks() if t["split"]==plan["split"]}
    requests = [{**slot, "messages":messages_for(views[slot["task_id"]]),
                 "generation":copy.deepcopy(plan["sampling"])} for slot in plan["slots"]]
    save_json(out/"plan.json",plan)
    save_jsonl(out/"requests.jsonl",requests)
    save_jsonl(out/"responses_template.jsonl",[
        make_record(s["task_id"],int(s["sample_id"][1:]),raw="",status="error",stop="error",base_seed=plan["base_seed"])
        for s in plan["slots"]])
    save_json(out/"contract.json", {"version":VERSION,"plan_sha256":digest(plan),"sources":SOURCES,
              "warning":"requests可发给生成器；评分源码含参考答案，不能作为生成输入。",
              "responses":"模板默认error，必须填实际输出、状态、计数，不能伪造成功。",
              "comparison":"first=首个随机样本，不是贪心基线；全部选择器复用同一候选池。"})


def write_evaluation(plan: dict, records: list[dict], out: Path, provenance: str) -> dict:
    validate_plan(plan)
    save_json(out/"plan.json",plan)
    save_jsonl(out/"candidates.jsonl",records)
    chosen = finalize_selection(plan,records)
    save_json(out/"selection.json",chosen)  # Commit the choice BEFORE private grading.
    report = score_selection(plan,records,chosen)
    save_json(out/"report.json",report)
    save_json(out/"provenance.json",{"label":provenance,"model_claim":provenance=="real_model_sampling",
                                    "selection_sha256":digest(chosen),"sources":SOURCES})
    return report


def replay_score(directory: str | Path, output_root: str | Path) -> tuple[Path,dict]:
    source = Path(directory).resolve()
    plan = load_json(source/"plan.json")
    records = load_jsonl(source/"candidates.jsonl")
    out = new_run(output_root,"rescore")
    report = write_evaluation(plan,records,out,"replayed_records_not_origin_authentication")
    return out,report


def default_experiment(output_root: str | Path = "lesson_56_outputs") -> tuple[Path,dict]:
    out = new_run(output_root,"offline")
    plan = build_plan("test",5)
    records = fixture_records(plan)
    export_requests(plan,out)
    report = write_evaluation(plan,records,out,"hand_authored_fixture_not_model_score")
    curves = []
    for n in (1,3,5):
        p = build_plan("test",n)
        r = fixture_records(p)
        s = finalize_selection(p,r)
        x = score_selection(p,r,s)
        curves.append({"n":n,"coverage":x["coverage"],"methods":x["methods"],"accounting":x["accounting"]})
    save_json(out/"budget_curves.json",curves)
    save_json(out/"mechanisms.json",mechanism_experiments())
    save_json(out/"manifest.json", {"version":VERSION,"mode":"offline_hand_authored",
             "python":platform.python_version(),"platform":platform.platform(),
             "real_model_executed":False,"cuda_executed":False,
             "source_dataset":"lesson55_fixed_4_dev_12_test_embedded",
             "next":"第57课：奖励信号与偏好数据——从评分到后训练目标"})
    return out,report


def run_real_sampling(output_root: str | Path, split: str = "dev", n: int = 5,
                      device: str = "cpu", local_files_only: bool = False) -> Path:
    """Optional, sequential Qwen sampling. No PEFT/training; no hidden answers in generation input.
    Not executed in the creation environment. Requires tested-by-user Transformers 4.57.3 + torch.
    """
    out = new_run(output_root,"real")
    plan = build_plan(split,n)
    export_requests(plan,out)
    manifest = {"mode":"real_model_sampling","status":"starting","model":MODEL_ID,
                "revision":MODEL_REVISION,"requested_device":device,"version":VERSION}
    save_json(out/"manifest.json",manifest)
    records = []
    stage = "imports"
    try:
        import torch
        import transformers
        from transformers import AutoTokenizer, AutoModelForCausalLM
        require(transformers.__version__ == "4.57.3", "本课真实入口固定transformers==4.57.3；请用独立环境")
        require(device in {"cpu","cuda"}, "device只接受cpu或cuda")
        require(device != "cuda" or torch.cuda.is_available(), "明确请求CUDA但不可用，不自动回退")
        dtype = torch.float16 if device=="cuda" else torch.float32
        kwargs = {"revision":MODEL_REVISION,"local_files_only":local_files_only,"trust_remote_code":False}
        stage = "load_tokenizer"
        tok = AutoTokenizer.from_pretrained(MODEL_ID,**kwargs)
        stage = "load_weights"
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=dtype,**kwargs).to(device)
        model.eval()
        manifest.update(torch_version=torch.__version__,transformers_version=transformers.__version__,
                        device=str(next(model.parameters()).device),dtype=str(next(model.parameters()).dtype),
                        device_name=torch.cuda.get_device_name(0) if device=="cuda" else platform.processor(),
                        template_sha256=digest(tok.chat_template))
        save_json(out/"manifest.json",manifest)
        views = {t["id"]:public_view(t) for t in make_tasks() if t["split"]==split}
        config = copy.deepcopy(model.generation_config)
        config.do_sample=True;config.temperature=0.8;config.top_p=0.95;config.top_k=20
        config.num_beams=1;config.num_return_sequences=1;config.repetition_penalty=1.0
        config.max_new_tokens=256;config.use_cache=True
        config.return_dict_in_generate=False;config.output_scores=False
        eos = config.eos_token_id if config.eos_token_id is not None else tok.eos_token_id
        eos_ids = list(eos) if isinstance(eos,(tuple,list)) else [eos]
        require(eos_ids and all(type(x) is int for x in eos_ids), "没有有效EOS设置")
        config.eos_token_id=eos_ids
        config.pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else eos_ids[0]
        save_json(out/"generation_config.json", config.to_dict())
        stage = "sampling"
        with (out/"candidates.jsonl").open("w",encoding="utf-8") as handle:
            for slot in plan["slots"]:
                tid=slot["task_id"];i=int(slot["sample_id"][1:])
                input_tokens=None
                try:
                    prompt=tok.apply_chat_template(messages_for(views[tid]),tokenize=False,
                                                   add_generation_prompt=True,enable_thinking=False)
                    inputs=tok(prompt,add_special_tokens=False,return_tensors="pt").to(device)
                    input_tokens=int(inputs["input_ids"].shape[1])
                    require(input_tokens+config.max_new_tokens<=plan["sampling"]["context_limit"],
                            "题目超过本课上下文预算；不静默截断")
                    # Set a new deterministic seed per task/slot; do NOT reset to the same seed every time.
                    torch.manual_seed(slot["seed"])
                    if device=="cuda": torch.cuda.manual_seed_all(slot["seed"]);torch.cuda.synchronize()
                    start=time.perf_counter()
                    with torch.inference_mode():
                        generated=model.generate(**inputs,generation_config=config)
                    if device=="cuda": torch.cuda.synchronize()
                    elapsed=(time.perf_counter()-start)*1000
                    new_ids=generated[0,input_tokens:].detach().cpu().tolist()
                    completed=bool(new_ids) and new_ids[-1] in eos_ids
                    text=tok.decode(new_ids,skip_special_tokens=True)
                    rec=make_record(tid,i,raw=text,stop="eos" if completed else "length",
                                    usage={"input_tokens":input_tokens,"output_tokens":len(new_ids),"latency_ms":elapsed})
                except Exception:
                    rec=make_record(tid,i,raw="",status="error",stop="error",
                                    usage={"input_tokens":input_tokens,"output_tokens":None,"latency_ms":None})
                    records.append(rec);handle.write(canonical(rec)+"\n");handle.flush()
                    raise  # Remaining slots stay missing. No hidden retries or fixture fallback.
                records.append(rec);handle.write(canonical(rec)+"\n");handle.flush()
                print(f"{tid}/{slot['sample_id']} {rec['stop_reason']} tokens={rec['usage']['output_tokens']}")
        chosen=finalize_selection(plan,records)
        save_json(out/"selection.json",chosen)
        save_json(out/"report.json",score_selection(plan,records,chosen))
        manifest.update(status="complete",received=len(records),weights_updated=False)
        save_json(out/"manifest.json",manifest)
        return out
    except Exception as exc:
        manifest.update(status="failed",stage=stage,error=f"{type(exc).__name__}: {exc}",received=len(records),
                        note="不会回退到手工记录。已保存的部分候选可单独重新评分。")
        save_json(out/"manifest.json",manifest)
        raise LessonError(f"真实路径失败，记录在 {out}: {exc}") from exc

In [18]:
RUN_DIR, REPORT = default_experiment(OUTPUT_ROOT)
print("本次输出：", RUN_DIR)
print("覆盖：", REPORT["coverage"])
for method, stats in REPORT["methods"].items():
    print(method, stats["correct"], "/", stats["planned"])
print("资源与缺失记录：")
print(json.dumps(REPORT["accounting"], ensure_ascii=False, indent=2))

本次输出： /mnt/data/.lesson56_build/notebook_run/lesson_56_outputs/offline_20261102T143514_accjxbbq
覆盖： {'correct': 12, 'planned': 12, 'rate': 1.0, 'label': 'observed_any_correct_not_deployable_or_iid_estimate'}
first 3 / 12
canonical_vote 5 / 12
public_best_of_n 6 / 12
资源与缺失记录：
{
  "planned_candidate_slots": 60,
  "received_records": 59,
  "missing_slots": 1,
  "complete_generation_records": 56,
  "known_output_tokens_sum": 0,
  "unknown_output_token_slots": 60,
  "all_slot_output_tokens": null,
  "public_code_cases_per_selection_pass": 36,
  "public_witness_conditions_per_selection_pass": 52,
  "selection_and_revalidation_passes": 2,
  "public_selection_case_evaluations_two_passes": 72,
  "private_grading_public_case_evaluations": 36,
  "private_grading_heldout_case_evaluations": 1936,
  "work_scope": "单次write_evaluation：公共选择及重校验分别计数；不含run_checks、预算曲线或其他重评分",
  "judge_model_calls": 0,
  "synthetic_model_latency_ms": null
}


### 5.1 三个指标必须分开

\[
\text{候选覆盖率}=\frac{\#\{\text{题目：池中至少有一条正确}\}}{\text{全部计划题目}}
\]

\[
\text{最终成功率}=\frac{\#\{\text{题目：所选候选正确}\}}{\text{全部计划题目}}
\]

\[
\text{有解池选择效率}=\frac{\text{最终选对的题数}}{\text{池中含正确答案的题数}}
\]

本次手工池覆盖 **12/12**，首样本、投票、公开代理选择分别为 **3/12、5/12、6/12**。覆盖上界只能用于评分后诊断，不能上线充当“自动选择正确答案”的实现。

一个没有正确候选的池，是生成侧瓶颈；池中有正确答案却没选中，是选择侧损失。两者可能同时需要改进。

In [19]:
for kind in ["exact", "quantity", "witness", "code"]:
    values = {name: row["by_kind"][kind] for name,row in REPORT["methods"].items()}
    print(kind, values)
print("代码池里有正确候选，但公开测试平分时仍选错。")

exact {'first': {'correct': 1, 'planned': 3}, 'canonical_vote': {'correct': 2, 'planned': 3}, 'public_best_of_n': {'correct': 1, 'planned': 3}}
quantity {'first': {'correct': 1, 'planned': 3}, 'canonical_vote': {'correct': 2, 'planned': 3}, 'public_best_of_n': {'correct': 2, 'planned': 3}}
witness {'first': {'correct': 1, 'planned': 3}, 'canonical_vote': {'correct': 1, 'planned': 3}, 'public_best_of_n': {'correct': 3, 'planned': 3}}
code {'first': {'correct': 0, 'planned': 3}, 'canonical_vote': {'correct': 0, 'planned': 3}, 'public_best_of_n': {'correct': 0, 'planned': 3}}
代码池里有正确候选，但公开测试平分时仍选错。


### 5.2 N=1、3、5 的预算曲线

只使用同一池的前 N 个计划位置，不按好坏重新抽样，不把缺失位置挤掉后补上第六条。

这三组手工前缀是**同池的预算回放**，不是重新进行三次独立模型实验；同一题上的结果相互相关。

In [20]:
curves = load_json(RUN_DIR / "budget_curves.json")
for curve in curves:
    print("N=", curve["n"],
          "coverage=", curve["coverage"]["correct"],
          "selected=", {k:v["correct"] for k,v in curve["methods"].items()})

N= 1 coverage= 3 selected= {'canonical_vote': 3, 'first': 3, 'public_best_of_n': 3}
N= 3 coverage= 12 selected= {'canonical_vote': 5, 'first': 3, 'public_best_of_n': 6}
N= 5 coverage= 12 selected= {'canonical_vote': 5, 'first': 3, 'public_best_of_n': 6}


## 6. 不要把观察到的覆盖率叫作部署 pass@k

在通常的抽样假设下，一题生成 n 个候选，其中 c 个通过，HumanEval 使用：

\[
\widehat{\mathrm{pass@k}}=1-\frac{\binom{n-c}{k}}{\binom{n}{k}}
\]

它估计候选中**至少有一个**能通过的可能性，不说明选择器知道是哪一个。[论文](https://arxiv.org/abs/2107.03374)｜[官方实现](https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py)

- `k=1` 的估计是 `c/n`，不是固定第一条是否通过。
- 缺失和异常如果计作失败，衡量的是整个运行流程；只保留已完成输出又会改变分母。
- 不能在 `k>n` 时无依据地外推。
- 本课手工池不是独立同分布的模型采样，**不把它的组合数结果宣传为模型 pass@k**。

不同种子也不保证错误原因独立。相同提示、相同模型常会共享盲点；跨硬件或版本也不保证逐位可复现。[PyTorch 复现说明](https://docs.pytorch.org/docs/stable/notes/randomness.html)

In [21]:
print("n=4,c=2,k=2 →", estimate_pass_at_k(4,2,2), "=5/6")
p, n = 0.6, 5
print("若每次成功相互独立，至少一次：", 1-(1-p)**n)
print("若五次成功事件完全相同，至少一次：", p)
print("这是概率假设的区别，不是本课测出了模型独立性。")

n=4,c=2,k=2 → 0.8333333333333334 =5/6
若每次成功相互独立，至少一次： 0.98976
若五次成功事件完全相同，至少一次： 0.6
这是概率假设的区别，不是本课测出了模型独立性。


## 7. 怎样增加真实多样性，而不是重复一次生成？

正温度采样允许多种输出，但不保证答案多样，更不保证正确。贪心重复运行通常不是五个独立随机样本；每次重复设置相同随机状态，也可能重现同一轨迹。

本课真实入口设置 `do_sample=True`、温度 0.8、`top_p=0.95`、`top_k=20`，每个计划槽使用不同派生 seed；**参数预先固定，未按测试成绩调节**。[生成参数文档](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)

下面只演示 Python 伪随机发生器的区别，不代表模型生成质量。

In [22]:
labels = ["candidate_A", "candidate_B", "candidate_C"]
reset_each_time = [random.Random(5600).choice(labels) for _ in range(5)]
seeded_slots = [random.Random(stable_seed("fraction_sum", i)).choice(labels) for i in range(5)]
print("每次同一seed：", reset_each_time)
print("按槽派生seed：", seeded_slots)
print("不同seed允许不同结果，但不能保证每次都不相同。")

每次同一seed： ['candidate_C', 'candidate_C', 'candidate_C', 'candidate_C', 'candidate_C']
按槽派生seed： ['candidate_A', 'candidate_A', 'candidate_C', 'candidate_B', 'candidate_B']
不同seed允许不同结果，但不能保证每次都不相同。


## 8. 成本包括生成、选择和失败；未知就保留未知

固定 N 相同，不代表实际成本相同。需要至少记录输入与输出 token、异常尝试、验证器开销、请求延迟，以及串行还是并行执行。

本课手工记录没有模型 token 或延迟，所以它们是 `null`，不是零。字符数不能拿来伪装成 tokenizer 的 token 数。

单次公共选择需要36个代码测试项和52个见证条件检查。为了验证选择记录，评分入口又进行一次公共重校验；私有最终评分也单独执行测试。报告把这些阶段分开计数，**36不是整个程序的总执行次数**。默认还会运行固定检查与预算曲线，它们不混入某一次选择的成本。这些是CPU评分工作，不是LLM verifier token。

假设每次生成输入100、输出40，另一个模型验证器输入160、输出10，共8个候选，那么总量应把两部分加起来。**这组数只是手算，不是实测；不同大小模型的token数也不等于等FLOPs。**

In [23]:
cost = mechanism_experiments()["hypothetical_cost"]
print("假设生成token：", cost["generation_tokens"])
print("假设验证token：", cost["verifier_tokens"])
print("假设总token：", cost["total_tokens"])
print("本次手工记录总模型token：", REPORT["accounting"]["all_slot_output_tokens"])
print("未知计数位置：", REPORT["accounting"]["unknown_output_token_slots"])

假设生成token： 1120
假设验证token： 1360
假设总token： 2480
本次手工记录总模型token： None
未知计数位置： 60


## 9. 可证明的固定上限早停，不是“相信多数一定对”

如果最多计划 N 票，已经观测 t 个位置，领先票数为 a、第二名为 b。若：

\[
a>b+(N-t)
\]

即使所有剩余票都投给竞争者，当前第一名也不会被反超。未出现的答案当前票数为零，同样被这个保守判据覆盖。

这只能证明**固定N的相对多数选择不会改变**，不能证明它正确。前三票一起答错，也可能提前锁定错误答案。

本课在已生成手工池上回放这个策略，得到47个观测位置、13个潜在可省位置。**池已完整存在，不宣称本次已经节省了13次实际模型调用。** 真正在线使用时，应先检查判据，再决定是否请求下一条。

In [24]:
selection = load_json(RUN_DIR / "selection.json")
used = sum(row["early_vote"]["consumed_slots"] for row in selection["rows"])
print("前缀回放读取位置：", used, "/", len(PLAN["slots"]))
print("潜在未请求位置：", len(PLAN["slots"])-used)
print("错误多数的停止点：", replay_early_vote(wrong_obs))

前缀回放读取位置： 47 / 60
潜在未请求位置： 13
错误多数的停止点： {'consumed_slots': 3, 'potential_unrequested_slots': 2, 'selected_index': 0, 'label': 'offline_prefix_replay_not_measured_generation_savings'}


## 10. 导出真实请求：先开发题，后固定测试

以下单元创建独立导出目录。`requests.jsonl` 可以交给第28课以后你已调通的模型接口；每条包含固定seed和生成设置，彼此不共享回答历史。

`responses_template.jsonl` 中的 `error` 是**待填写状态**。应填入真实输出与停止原因，不能复制参考答案或统一填写 `eos`。

评分源码公开包含答案，但**不要把整份脚本、参考实现、额外测试或评分报告作为模型上下文**。

In [25]:
EXPORT_DIR = new_run(OUTPUT_ROOT, "export_dev")
export_requests(build_plan("dev", n=3), EXPORT_DIR)
print("导出目录：", EXPORT_DIR)
first_request = load_jsonl(EXPORT_DIR / "requests.jsonl")[0]
print("请求中的字段：", sorted(first_request))
print("首条消息：", first_request["messages"])

导出目录： /mnt/data/.lesson56_build/notebook_run/lesson_56_outputs/export_dev_20261102T143514_5c151tj6
请求中的字段： ['generation', 'messages', 'request_sha256', 'sample_id', 'seed', 'task_id']
首条消息： [{'content': '遵守题目给定的输出格式。只返回最终JSON，不附加解释或代码围栏。', 'role': 'system'}, {'content': '计算 3/4 + 1/8。\n仅输出JSON {"answer":"数值"}；数值是整数、普通十进制小数或分数的字符串，合计最多24个数字；保留正负号，不用指数、百分号、单位、解释或代码围栏。精确比较，不四舍五入。', 'role': 'user'}]


### 10.1 可选：顺序调用真实 Qwen3-0.6B

代码固定模型 commit，使用配套 Chat Template，`enable_thinking=False`，只取新增输出。Qwen3支持这一模板开关。[模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)

默认每次只生成一个候选，不同时驻留N份KV cache；N增加会增加工作量，但并不要求复制N份权重。输入、输出和任何错误都会逐条落盘，未完成位置保持缺失。没有隐式重试，没有手工夹具回退。

在你已有可用的PyTorch环境中，缺少依赖时安装课程固定接口：

```bash
python -m pip install "transformers==4.57.3" safetensors
```

```bash
python lesson_56_sampling_self_consistency_and_best_of_n.py --sample --split dev --n 5 --device cuda
```

设置确定后才执行 `--split test`。CPU使用`--device cpu`；缓存完整后追加`--local-files-only`。

**制作环境未安装Transformers，真实采样未端到端实测。** 这不是最新版声明或已验证的CUDA兼容组合。明确请求CUDA不可用时会报错；不自动回退。首条候选仍是随机样本，不是贪心。

In [26]:
RUN_REAL_SAMPLING = False
REAL_DEVICE = "cpu"
REAL_SPLIT = "dev"
REAL_N = 3

if RUN_REAL_SAMPLING:
    real_dir = run_real_sampling(OUTPUT_ROOT, split=REAL_SPLIT, n=REAL_N, device=REAL_DEVICE)
    print("真实记录目录：", real_dir)
else:
    print("真实采样未执行；以上结果均来自手工候选。")

真实采样未执行；以上结果均来自手工候选。


### 10.2 导入回答与重新评分

```bash
python lesson_56_sampling_self_consistency_and_best_of_n.py --grade my_responses.jsonl --plan 导出目录/plan.json
```

```bash
python lesson_56_sampling_self_consistency_and_best_of_n.py --rescore 本课实际运行目录
```

重新评分只读原计划与原始候选，重新计算公共选择和私有评分，不信任旧报告中的通过标记，也不会覆盖原目录。

**协议指纹不能证明数据来源诚实。** 导入外部响应时，模型版本、模板和解码配置仍要由实验者如实记录。不能混合不同设置后只贴同一个seed。

不要一边看测试评分一边调整选择器阈值，再声称仍然是盲测；本课题集已经是公开课堂回归材料。

In [27]:
RESCORE_DIR, REPLAY_REPORT = replay_score(RUN_DIR, OUTPUT_ROOT)
print("新评分目录：", RESCORE_DIR)
print("和原报告完全一致：", REPLAY_REPORT == REPORT)

新评分目录： /mnt/data/.lesson56_build/notebook_run/lesson_56_outputs/rescore_20261102T143514_0ggwat48
和原报告完全一致： True


## 11. 独立练习：先停下来自己写

下面只装入检查函数。参考答案在后面，使用不同函数名，不会覆盖你的作业。

**A：精确规范化。** 支持有界整数、小数、分数；等价数值归为同一key；不接受float、多个答案、零分母或NaN；不要把两个本来不相等的精确数值按浮点舍入合并。

In [28]:
def check_canonical_ex(fn) -> dict:
    for x,e in [("1/2","1/2"),("0.5","1/2"),("2/4","1/2"),("-0","0"),("-14","-14")]:
        require(fn(x)==e,"规范化结果错误")
    for bad in [True,0.5,"1/0","14 or 140","NaN","1e-3"]:
        try: fn(bad)
        except (LessonError,ValueError,TypeError): pass
        else: raise AssertionError("非法输入没有拒绝")
    require(fn("0.30000000000000004") != fn("0.3"),"精确比较不应偷偷容差合并")
    return {"exercise":"A","passed":True}

def check_vote_ex(fn) -> dict:
    def obs(keys):
        return [{"index":i,"key":k,"eligible":k is not None} for i,k in enumerate(keys)]
    for keys,expected in [(["a","b","b"],1),(["b","a"],0),([None,"a","a"],1),
                          ([None,None],None),(["a","a","b","b","b"],2)]:
        x=obs(keys);before=copy.deepcopy(x)
        require(fn(x)==expected,"票数/并列/空集处理错误")
        require(x==before,"不应修改观察记录")
    return {"exercise":"B","passed":True}

def check_budget_ex(fn) -> dict:
    require(fn([True,True,False],4,[10,None,20])==
            {"rate":0.5,"missing_results":1,"known_cost":30,"unknown_cost_slots":2,"total_cost":None},
            "不能删除失败/缺失，也不能把未知成本当0")
    require(fn([True],1,[5])["total_cost"]==5,"完整成本错误")
    try: fn([True,True],1,[5])
    except (LessonError,ValueError): pass
    else: raise AssertionError("没有拒绝计划外记录")
    return {"exercise":"C","passed":True}

In [29]:
def canonical_rational_ex(answer):
    # TODO：检查输入，使用 Fraction 从原始字符串创建，返回规范字符串。
    raise NotImplementedError("请完成练习 A")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(check_canonical_ex(canonical_rational_ex))

**B：相对多数投票。** 输入是已经规范化的观察列表，返回原始计划索引或None。排除不合法响应；相同答案的独立样本不能去重；同票取最早索引；不要修改原列表。

思考：如果五个计划位置只有一条合法回答，你会报告怎样的有效票一致率与计划槽支持率？

In [30]:
def vote_ex(observations):
    # TODO：分组计票；按票数降序、最早索引升序选择。
    raise NotImplementedError("请完成练习 B")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(check_vote_ex(vote_ex))

**C：固定分母与未知成本。** 计划4次，收到3个成功标记`[True,True,False]`，成本记录为`[10,None,20]`。成功率应该是1/2，不是2/3；已知成本为30，但完整成本为None，另有两个成本未知的位置。

不要把缺失记录排除，也不要把未知成本写成0。

In [31]:
def budget_ex(success_flags, planned, costs):
    # TODO：校验数量与类型，分别返回已知成本和总成本是否可知。
    raise NotImplementedError("请完成练习 C")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(check_budget_ex(budget_ex))

---
## 12. 参考答案与验收

完成自己的实现后再展开。这里采用与主程序一致的受限解析和明确约定，避免为了过检查进行模糊匹配。

In [32]:
def canonical_rational_ex_reference(answer: str) -> str:
    return str(parse_rational(answer))

def vote_ex_reference(observations: list[dict]) -> int | None:
    return select_vote(observations)["selected_index"]

def budget_ex_reference(success_flags: list[bool], planned: int, costs: list[int | None]) -> dict:
    require(type(planned) is int and planned>0, "planned必须是正整数")
    require(len(success_flags)<=planned and len(costs)<=planned, "记录数超过计划")
    require(all(type(x) is bool for x in success_flags), "成功标记必须是bool")
    require(all(x is None or (type(x) is int and x>=0) for x in costs), "成本计数非法")
    known=[x for x in costs if x is not None]
    return {"rate":sum(success_flags)/planned,"missing_results":planned-len(success_flags),
            "known_cost":sum(known),"unknown_cost_slots":planned-len(known),
            "total_cost":sum(known) if len(known)==planned else None}

In [33]:
print(check_canonical_ex(canonical_rational_ex_reference))
print(check_vote_ex(vote_ex_reference))
print(check_budget_ex(budget_ex_reference))

{'exercise': 'A', 'passed': True}
{'exercise': 'B', 'passed': True}
{'exercise': 'C', 'passed': True}


### 自动检查与故障注入

固定检查包括公共视图不含答案、选择阶段不调用私有评分器、记录顺序无关性、错误多数、样本重复计票、公开代码测试盲点、缺失分母、完整落盘重评分以及243条短投票轨迹的早停一致性。

故障注入会临时替换实现，并用同一套检查检测：先转float、投票去重、代理排序方向错误、接受截断输出、早停忽略剩余票、未知成本填零。

测试通过说明这些检查覆盖了已列出的行为，不代表所有漏洞都被排除。

In [34]:
def run_checks() -> dict:
    names=[]
    def check(name, condition):
        require(condition, "检查失败："+name)
        names.append(name)
    def rejects(name, fn):
        try: fn()
        except (LessonError, ValueError, TypeError): names.append(name)
        else: raise AssertionError("应该拒绝："+name)
    tasks=make_tasks();by={t["id"]:t for t in tasks}
    plan=build_plan();records=fixture_records(plan)
    chosen=finalize_selection(plan,records)
    report=score_selection(plan,records,chosen)
    fv=public_view(by["fraction_sum"])
    def observe_values(values):
        return [observe_candidate(fv,make_record("fraction_sum",i,{"answer":x}),i) for i,x in enumerate(values)]
    check("完整保留第55课4道开发与12道测试",Counter(t["split"] for t in tasks)=={"dev":4,"test":12})
    check("四种任务各三道",Counter(t["kind"] for t in tasks if t["split"]=="test")==dict.fromkeys(["exact","quantity","witness","code"],3))
    check("五次尝试预登记60个位置",len(plan["slots"])==60)
    check("种子稳定且与遍历次序无关",stable_seed("duration",2)==stable_seed("duration",2))
    check("不同样本不重置成同一随机种子",stable_seed("duration",0)!=stable_seed("duration",1))
    check("不同题目有独立派生种子",stable_seed("duration",0)!=stable_seed("length",0))
    rejects("拒绝零候选预算",lambda:build_plan(n=0))
    rejects("拒绝bool候选预算",lambda:build_plan(n=True))
    rejects("拒绝过大候选预算",lambda:build_plan(n=100))
    changed=copy.deepcopy(plan);changed["sampling"]["temperature"]=1.2
    rejects("采样设置变更需要新契约",lambda:validate_plan(changed))
    check("生成视图无数学参考答案","expected" not in public_view(by["duration"]))
    check("代码请求不含额外评测用例","evaluation_cases" not in public_view(by["code_count_nonzero"]))
    check("代码公开样例保留可用反馈",len(public_view(by["code_count_nonzero"])["public_examples"])==3)
    injected=copy.deepcopy(fv);injected["expected"]="1/2"
    rejects("选择器拒绝混入私有目标的视图",lambda:observe_candidate(injected,records[0],0))
    check("等价分数与小数合并",len({o["key"] for o in observe_values(["1/2","0.5","2/4"])})==1)
    check("保留负号",observe_values(["-14"])[0]["key"]!=observe_values(["14"])[0]["key"])
    check("不按子串判答案",observe_values(["14"])[0]["key"]!=observe_values(["140"])[0]["key"])
    check("不隐含浮点容差",observe_values(["0.3"])[0]["key"]!=observe_values(["0.30000000000000004"])[0]["key"])
    rejects("不接受模糊多答案",lambda:parse_rational("14 or 140"))
    rejects("不接受零分母",lambda:parse_rational("1/0"))
    rejects("不接受JSON重复键",lambda:strict_json('{"answer":"1","answer":"2"}'))
    rejects("不接受NaN",lambda:strict_json('{"answer":NaN}'))
    check("单位换算等价",normalize_quantity("90","min")==normalize_quantity("1.5","h"))
    check("单位种类不能合并",normalize_quantity("90","m")!=normalize_quantity("90","min"))
    wit=public_view(by["subset_positive"])
    w1=observe_candidate(wit,make_record(wit["task_id"],0,{"items":[1,4,7]}),0)
    w2=observe_candidate(wit,make_record(wit["task_id"],1,{"items":[7,4,1]}),1)
    check("见证排列不分票",w1["key"]==w2["key"])
    w3=observe_candidate(wit,make_record(wit["task_id"],2,{"items":[2,4,6]}),2)
    check("不同合法见证允许分成不同组",w1["proxy_score"]==w3["proxy_score"]==1 and w1["key"]!=w3["key"])
    check("重复元素不能当成合法见证",observe_candidate(wit,make_record(wit["task_id"],3,{"items":[4,4,4]}),3)["proxy_score"]<1)
    check("合法但数值错误仍可能取得格式分",observe_values(["999"])[0]["proxy_score"]==1)
    malformed=make_record("fraction_sum",0,raw="答案是1/2")
    check("格式非法不投票",not observe_candidate(fv,malformed,0)["eligible"])
    truncated=make_record("fraction_sum",0,{"answer":"1/2"},stop="length")
    check("截断结果不投票",not observe_candidate(fv,truncated,0)["eligible"])
    check("缺失结果保留空槽",observe_candidate(fv,None,3)["reason"]=="missing")
    check("多数票不去重采样",select_vote(observe_values(["1","2","2"]))["selected_index"]==1)
    check("同票取最早计划位置",select_vote(observe_values(["2","1"]))["selected_index"]==0)
    empty=[observe_candidate(fv,None,i) for i in range(5)]
    check("全部无效时投票拒答",select_vote(empty)["selected_index"] is None)
    check("全部无效时Best-of-N拒答",select_best(empty) is None)
    check("首样本不自动换成首个成功者",select_first(empty)==0)
    one=[observe_values(["1"])[0]]+empty[1:]
    check("有效票一致率与计划分母分开",select_vote(one)["agreement_valid"]==1 and select_vote(one)["agreement_planned"]==0.2)
    check("投票一致不保证正确",mechanism_experiments()["confidently_wrong"]["vote"]["votes"]==3)
    cv=public_view(by["code_count_nonzero"])
    co=[observe_candidate(cv,make_record(cv["task_id"],i,{"code":s}),i) for i,s in enumerate([WRONG_CODE["count_nonzero"],CORRECT_CODE["count_nonzero"]])]
    check("代码公开样例盲点",co[0]["proxy_score"]==co[1]["proxy_score"]==1)
    check("代码行为投票不等于语义证明",co[0]["key"]==co[1]["key"])
    check("Best-of-N平分不偷看额外测试",select_best(co)==0)
    check("独立额外测试发现假阳性",not grade_candidate(by["code_count_nonzero"],make_record(cv["task_id"],0,{"code":WRONG_CODE["count_nonzero"]}))["strict_success"])
    check("参考代码通过有限测试",grade_candidate(by["code_count_nonzero"],make_record(cv["task_id"],0,{"code":CORRECT_CODE["count_nonzero"]}))["strict_success"])
    check("禁止执行函数调用语法",not observe_candidate(cv,make_record(cv["task_id"],0,{"code":"def solve(numbers):\n    return len(numbers)\n"}),0)["eligible"])
    rejects("禁止import",lambda:parse_program("import os\ndef solve(numbers):\n    return 0\n"))
    before=copy.deepcopy(co);select_best(co);select_vote(co)
    check("选择器不改写候选观察",co==before)
    check("无隐藏答案可完成选择",chosen["gold_used"] is False)
    # Mutation of all private targets must leave public selectors unchanged.
    original_make=globals()["make_tasks"]
    altered=copy.deepcopy(tasks)
    for t in altered:
        if "expected" in t: t["expected"]="999999"
        if "evaluation_cases" in t:
            for c in t["evaluation_cases"]: c["expected"]=-999999
    # Compare public observations directly: plan hashes intentionally change with a new private task set.
    aviews={t["id"]:public_view(t) for t in altered}
    check("修改私有答案不改变公共视图",all(aviews[t["id"]]==public_view(t) for t in tasks))
    original_grade=globals()["grade_candidate"]
    def forbidden(*a,**kw): raise AssertionError("选择阶段调用了私有评分器")
    try:
        globals()["grade_candidate"]=forbidden
        check("选择阶段不调用最终评分",finalize_selection(plan,records)==chosen)
    finally:globals()["grade_candidate"]=original_grade
    check("手工响应有一个缺失计划槽",len(records)==59 and report["accounting"]["missing_slots"]==1)
    check("覆盖上界12题",report["coverage"]["correct"]==12)
    check("首样本3题",report["methods"]["first"]["correct"]==3)
    check("投票5题",report["methods"]["canonical_vote"]["correct"]==5)
    check("公开代理选择6题",report["methods"]["public_best_of_n"]["correct"]==6)
    check("四类任务分母各3",all(g["planned"]==3 for m in report["methods"].values() for g in m["by_kind"].values()))
    check("所有选择成功不超过覆盖",all(m["correct"]<=report["coverage"]["correct"] for m in report["methods"].values()))
    check("未知token成本不填零",report["accounting"]["all_slot_output_tokens"] is None)
    check("每次公共选择需要36个代码测试项",report["accounting"]["public_code_cases_per_selection_pass"]==36)
    check("公开见证检查52项",report["accounting"]["public_witness_conditions_per_selection_pass"]==52)
    check("记录顺序不改变选择",finalize_selection(plan,list(reversed(records)))==chosen)
    rejects("重复ID不能重复投票",lambda:index_records(plan,records+[records[0]]))
    wrong=copy.deepcopy(records);wrong[0]["seed"]+=1
    rejects("身份中的seed不能悄悄替换",lambda:index_records(plan,wrong))
    wrong=copy.deepcopy(records);wrong[0]["sample_id"]="s12"
    rejects("不能混入预算之外候选",lambda:index_records(plan,wrong))
    wrong=copy.deepcopy(records);wrong[0]["proxy_score"]=1.0
    rejects("模型自报分数不被选择器接受",lambda:index_records(plan,wrong))
    wrong=copy.deepcopy(records);wrong[0]["usage"]["output_tokens"]=True
    rejects("bool不算token计数",lambda:index_records(plan,wrong))
    forged=copy.deepcopy(chosen);forged["rows"][0]["selected"]["first"]=1
    rejects("评分前拒绝篡改选择结果",lambda:score_selection(plan,records,forged))
    check("空记录仍保留12题分母",score_selection(plan,[],finalize_selection(plan,[]))["methods"]["first"]["planned"]==12)
    check("空记录成功率为0而非删题",score_selection(plan,[],finalize_selection(plan,[]))["coverage"]["rate"]==0)
    check("pass@k组合数例子",math.isclose(estimate_pass_at_k(4,2,2),5/6))
    check("没有正确候选的pass@k",estimate_pass_at_k(5,0,3)==0)
    check("全部正确的pass@k",estimate_pass_at_k(5,5,3)==1)
    check("k=n且至少一个正确",estimate_pass_at_k(5,1,5)==1)
    rejects("pass@k不能外推k>n",lambda:estimate_pass_at_k(4,2,5))
    rejects("pass@k拒绝bool",lambda:estimate_pass_at_k(True,1,1))
    check("三比零且只剩二票可锁定",vote_locked(observe_values(["1","1","1"]),2))
    check("两比一且仍剩二票不能锁定",not vote_locked(observe_values(["1","1","2"]),2))
    check("没有有效票不能锁定",not vote_locked(empty,0))
    require(all(replay_early_vote(r["observations"])["selected_index"]==r["vote"]["selected_index"] for r in chosen["rows"]),"早停与完整票池不一致")
    check("固定上限的提前锁票不改变这组选择",True)
    check("早停只重放47槽不伪称已少生成",sum(r["early_vote"]["consumed_slots"] for r in chosen["rows"])==47)
    for sequence in itertools.product([None,"a","b"],repeat=5):
        oo=[{"index":i,"key":k,"eligible":k is not None} for i,k in enumerate(sequence)]
        require(replay_early_vote(oo)["selected_index"]==select_vote(oo)["selected_index"],"早停性质失败")
    check("全部243条小型投票轨迹提前锁票一致",True)
    check("加入高代理分错解可让Best-of-N退步",mechanism_experiments()["nonmonotonic_best_of_n"]==
          {"N1_selected_correct":True,"N2_selected_correct":False,"label":"手工代理分数反例，不是学习到的验证器"})
    check("独立假设与完全一致事件不能混用",mechanism_experiments()["independence_example"]["iid_any"]>0.98)
    check("成本包括生成与验证",mechanism_experiments()["hypothetical_cost"]["total_tokens"]==2480)
    check_canonical_ex(canonical_rational_ex_reference);names.append("练习A参考实现")
    check_vote_ex(vote_ex_reference);names.append("练习B参考实现")
    check_budget_ex(budget_ex_reference);names.append("练习C参考实现")
    with tempfile.TemporaryDirectory() as tmp:
        d=Path(tmp)
        export_requests(build_plan("dev",2),d)
        req=load_jsonl(d/"requests.jsonl")
        check("导出4道开发题每题2次",len(req)==8)
        check("每条请求独立且没有答案字段",all(set(r)=={"task_id","sample_id","seed","request_sha256","messages","generation"} for r in req))
        check("导出模板保持error不伪造成功",all(r["status"]=="error" for r in load_jsonl(d/"responses_template.jsonl")))
        out=d/"eval";out.mkdir()
        written=write_evaluation(plan,records,out,"test_fixture")
        check("选择文件与评分绑定",load_json(out/"report.json")["selection_sha256"]==digest(load_json(out/"selection.json")))
        before={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir()}
        _,r2=replay_score(out,d/"new")
        check("落盘重评分一致",r2==written)
        check("重评分不改写原始文件",before=={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir()})
        bad=d/"bad.json";bad.write_text('{"x":1,"x":2}',encoding="utf-8")
        rejects("文件级重复键拒绝",lambda:load_json(bad))
    return {"passed":len(names),"checks":names,"scope":"手工候选、有限解释器与协议；不含真实模型或GPU实测"}


@contextmanager
def patched_global(name: str, replacement):
    old=globals()[name];globals()[name]=replacement
    try:yield
    finally:globals()[name]=old


def mutation_checks() -> dict:
    """Do the unchanged checks catch intentionally broken implementation variants?"""
    original_parse=parse_rational
    def approximate_parse(s):
        return Fraction(float(original_parse(s)))
    original_vote=select_vote
    def deduplicating_vote(obs):
        seen=set(); changed=copy.deepcopy(obs)
        for x in changed:
            if x["eligible"]:
                if x["key"] in seen:x["eligible"]=False
                else:seen.add(x["key"])
        return original_vote(changed)
    def wrong_best(obs):
        eligible=[o for o in obs if o["eligible"]]
        return None if not eligible else min(eligible,key=lambda x:(x["proxy_score"],x["index"]))["index"]
    original_observe=observe_candidate
    def allow_truncated(view,record,index):
        if record is not None:
            record=copy.deepcopy(record)
            if record["status"]=="ok":record["stop_reason"]="eos"
        return original_observe(view,record,index)
    def ignore_early_remaining(obs,remaining):
        return any(x["eligible"] for x in obs)
    original_score=score_selection
    def replace_unknown_cost_with_zero(*args):
        result=original_score(*args)
        result["accounting"]["all_slot_output_tokens"]=result["accounting"]["known_output_tokens_sum"]
        return result
    variants=[("数值先转float","parse_rational",approximate_parse),
              ("把相同答案去重再投票","select_vote",deduplicating_vote),
              ("Best-of-N排序方向反了","select_best",wrong_best),
              ("截断输出也参与选择","observe_candidate",allow_truncated),
              ("提前停止忽略剩余票数","vote_locked",ignore_early_remaining),
              ("未知token用量记为零","score_selection",replace_unknown_cost_with_zero)]
    results=[]
    for label,name,broken in variants:
        detected=False;error=None
        with patched_global(name,broken):
            try:run_checks()
            except Exception as exc:detected=True;error=f"{type(exc).__name__}: {exc}"
        require(detected,"故障未被发现："+label)
        results.append({"mutation":label,"detected":detected,"first_failure":error})
    return {"detected":len(results),"mutations":results}


def print_summary(report: dict) -> None:
    print(f"N={report['n']}，候选池覆盖：{report['coverage']['correct']}/{report['coverage']['planned']}")
    for name,item in report["methods"].items():
        print(f"{name}: {item['correct']}/{item['planned']}; selection_misses={item['selection_misses']}")


def main(argv: list[str] | None = None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    modes=parser.add_mutually_exclusive_group()
    modes.add_argument("--checks-only",action="store_true")
    modes.add_argument("--math-only",action="store_true")
    modes.add_argument("--export-prompts",action="store_true")
    modes.add_argument("--sample",action="store_true",help="可选真实Qwen采样；无教学结果回退")
    modes.add_argument("--grade",type=Path,help="导入真实响应JSONL；需要--plan")
    modes.add_argument("--rescore",type=Path,help="本课包含plan.json、candidates.jsonl的运行目录")
    modes.add_argument("--doctor",action="store_true")
    parser.add_argument("--plan",type=Path)
    parser.add_argument("--split",choices=["dev","test"],default="dev")
    parser.add_argument("--n",type=int,default=5)
    parser.add_argument("--device",choices=["cpu","cuda"],default="cpu")
    parser.add_argument("--local-files-only",action="store_true")
    parser.add_argument("--output-dir",type=Path,default=Path("lesson_56_outputs"))
    args=parser.parse_args(argv)
    if args.doctor:
        import importlib.util
        print(canonical({"python":platform.python_version(),"platform":platform.platform(),
                         "torch_installed":importlib.util.find_spec("torch") is not None,
                         "transformers_installed":importlib.util.find_spec("transformers") is not None,
                         "default_requires":"Python standard library only"}))
        return 0
    if args.checks_only:
        checks=run_checks()
        print(f"固定检查通过：{checks['passed']}项。手工候选/协议检查，不是模型成绩。")
        return 0
    if args.math_only:
        print(json.dumps(mechanism_experiments(),ensure_ascii=False,indent=2));return 0
    if args.export_prompts:
        out=new_run(args.output_dir,"export");export_requests(build_plan(args.split,args.n),out)
        print("导出：",out);return 0
    if args.sample:
        out=run_real_sampling(args.output_dir,args.split,args.n,args.device,args.local_files_only)
        print("真实采样记录：",out);return 0
    if args.grade:
        require(args.plan is not None,"--grade需要--plan指向事先导出的计划")
        out=new_run(args.output_dir,"imported")
        report=write_evaluation(load_json(args.plan),load_jsonl(args.grade),out,"external_records_not_authenticated")
        print_summary(report);print("报告：",out);return 0
    if args.rescore:
        out,report=replay_score(args.rescore,args.output_dir)
        print_summary(report);print("新报告：",out);return 0
    checks=run_checks();mutations=mutation_checks()
    out,report=default_experiment(args.output_dir)
    save_json(out/"checks.json",checks);save_json(out/"mutations.json",mutations)
    print(f"固定检查通过：{checks['passed']}项；故障注入检出：{mutations['detected']}种。")
    print("以下为手工候选夹具，不是模型成绩：")
    print_summary(report);print("报告目录：",out)
    return 0

In [35]:
CHECKS = run_checks()
MUTATIONS = mutation_checks()
save_json(RUN_DIR / "checks.json", CHECKS)
save_json(RUN_DIR / "mutations.json", MUTATIONS)
print("固定检查通过：", CHECKS["passed"])
print("故障注入检出：", MUTATIONS["detected"])
print("真实模型和GPU：未在默认流程运行")

固定检查通过： 93
故障注入检出： 6
真实模型和GPU：未在默认流程运行


## 13. 带走的结论与下一课

**候选中有解，不等于选择器选对；答案一致，不等于答案正确；增加候选，不等于免费提升。**

本课完成标准：能解释本次12/12覆盖为何只能得到5/12投票成功和6/12代理选择成功；能指出评分信息边界；能在另一进程中从原始候选重新算出相同结果；完成三道独立练习。

下一课：**第57课：奖励信号、偏好数据与后训练目标。** 从可验证结果与候选对开始，区分奖励、偏好和监督目标，再逐步理解DPO等方法。不会把本课代理分数自动视为真实用户偏好。

### 原始资料

1. [Self-Consistency 原论文](https://arxiv.org/abs/2203.11171)：多路径采样与最终答案聚合。
2. [Training Verifiers](https://arxiv.org/abs/2110.14168)：采样多个候选，再使用验证器排序。
3. [Codex / HumanEval 论文](https://arxiv.org/abs/2107.03374)与[官方pass@k实现](https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py)：至少一个正确候选的估计。
4. [测试时计算分配研究](https://arxiv.org/abs/2408.03314)：计算分配与题目难度的关系。
5. [Python Fraction](https://docs.python.org/3.13/library/fractions.html)：精确有理数处理。
6. [Transformers 4.57.3 生成参数](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)与[Qwen固定revision](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)：可选真实运行接口。

本课不依赖第三方服务、不上传数据。输出中包含原始答案与候选代码，分享前检查是否有私人资料。